In [3]:
# Step 1 — Clone the research repository
!git clone -b v2-research https://github.com/kharoojn274/3d-RCNN-using-Unet-for-brain-tumor-segmentation.git

Cloning into '3d-RCNN-using-Unet-for-brain-tumor-segmentation'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 131 (delta 51), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 2.72 MiB | 14.35 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
%cd /kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation

/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation


In [3]:
!find v2 -maxdepth 3 -type f

v2/data/dataset.py
v2/data/brats2020_dataset.py
v2/data/preprocessing.py
v2/data/augmentation.py
v2/data/verify_split_and_pipeline.py
v2/data/brats_gli_dataset.py
v2/training/config.py
v2/training/train.py
v2/training/losses.py
v2/training/engine.py
v2/training/metrics.py
v2/models/blocks.py
v2/models/rcnn_unet3d.py


In [6]:
# Step 2 — GPU check
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [7]:
# Step 3 — Check required packages
!pip install -q nibabel scipy scikit-learn

In [8]:
import nibabel
import scipy
import sklearn

print("nibabel:", nibabel.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)

nibabel: 5.4.2
scipy: 1.16.3
sklearn: 1.6.1


In [9]:
# Step 4 — Set the repository path and import the V2 code
%cd /kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation

import sys
sys.path.insert(
    0,
    "/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation"
)

from v2.data import brats2020_dataset
from v2.data import preprocessing
from v2.data import augmentation

from v2.models.rcnn_unet3d import RCNNUNet3D
from v2.training.losses import DiceCrossEntropyLoss
from v2.training.metrics import *
from v2.training.config import *

print("✅ Existing V2 modules imported successfully")

/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation
✅ Existing V2 modules imported successfully


In [10]:
# Step 5 — Verify the BraTS 2020 dataset
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/"
    "brats20-dataset-training-validation/"
    "BraTS2020_TrainingData/"
    "MICCAI_BraTS2020_TrainingData"
)

print("Dataset root:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())

subjects = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("BraTS20_Training_")
)

print("Number of subject directories:", len(subjects))
print("First 5 subjects:")

for subject in subjects[:5]:
    print(" ", subject.name)

Dataset root: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
Exists: True
Number of subject directories: 369
First 5 subjects:
  BraTS20_Training_001
  BraTS20_Training_002
  BraTS20_Training_003
  BraTS20_Training_004
  BraTS20_Training_005


In [12]:
# Step 6 — Check all 369 subjects for missing files
from pathlib import Path

MODALITIES = ("t1", "t1ce", "t2", "flair")

def find_modality_file(subject_dir, subject_id, modality):
    nii = subject_dir / f"{subject_id}_{modality}.nii"
    niigz = subject_dir / f"{subject_id}_{modality}.nii.gz"

    if nii.exists():
        return nii
    if niigz.exists():
        return niigz

    return None


def find_segmentation_file(subject_dir, subject_id):
    # Standard BraTS filename
    standard_names = [
        f"{subject_id}_seg.nii",
        f"{subject_id}_seg.nii.gz",
    ]

    for name in standard_names:
        path = subject_dir / name
        if path.exists():
            return path

    # Some Kaggle copies may contain a differently named segmentation.
    # We identify it conservatively by filename containing "seg".
    candidates = [
        p for p in subject_dir.iterdir()
        if p.is_file()
        and p.suffix in (".nii", ".gz")
        and "seg" in p.name.lower()
    ]

    if len(candidates) == 1:
        return candidates[0]

    return None


problems = []

for subject_dir in subjects:
    subject_id = subject_dir.name
    missing = []

    for modality in MODALITIES:
        if find_modality_file(subject_dir, subject_id, modality) is None:
            missing.append(modality)

    if find_segmentation_file(subject_dir, subject_id) is None:
        missing.append("seg")

    if missing:
        problems.append({
            "subject": subject_id,
            "missing": missing,
            "files": sorted(p.name for p in subject_dir.iterdir())
        })


print("Total subjects checked:", len(subjects))
print("Subjects with problems:", len(problems))

for item in problems:
    print("\nSubject:", item["subject"])
    print("Missing:", item["missing"])
    print("Files:")
    for filename in item["files"]:
        print(" ", filename)

Total subjects checked: 369
Subjects with problems: 0


In [13]:
# Step 7 — Test the current dataset loader
from pathlib import Path

dataset_file = Path(
    "/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation/"
    "v2/data/brats2020_dataset.py"
)

code = r'''
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence

import nibabel as nib
import numpy as np
import torch
from torch.utils.data import Dataset


MODALITIES = ("t1", "t1ce", "t2", "flair")
LABEL_NAME = "seg"

# Original BraTS 2020 labels
LABELS = {
    "background": 0,
    "necrosis": 1,
    "edema": 2,
    "enhancing_tumor": 4,
}


def find_file(
    subject_dir: Path,
    subject_id: str,
    suffix: str,
) -> Path:
    """
    Find a BraTS file supporting both .nii and .nii.gz.

    For segmentation, BraTS20_Training_355 in this Kaggle dataset
    has a non-standard filename:
        W39_1998.09.19_Segm.nii

    Therefore segmentation discovery also searches for a unique
    filename containing 'seg' or 'segm'.
    """

    # Standard filename
    nii = subject_dir / f"{subject_id}_{suffix}.nii"
    niigz = subject_dir / f"{subject_id}_{suffix}.nii.gz"

    if nii.is_file():
        return nii

    if niigz.is_file():
        return niigz

    # Special handling for segmentation files with non-standard names.
    if suffix == LABEL_NAME:
        candidates = sorted(
            p for p in subject_dir.iterdir()
            if p.is_file()
            and p.suffix in {".nii", ".gz"}
            and (
                "seg" in p.name.lower()
                or "segm" in p.name.lower()
            )
        )

        if len(candidates) == 1:
            return candidates[0]

    raise FileNotFoundError(
        f"Missing {subject_id}_{suffix}.nii/.nii.gz "
        f"in {subject_dir}"
    )


def discover_subjects(root: str | Path) -> List[Path]:
    """
    Discover valid BraTS 2020 training subjects.

    Supports both:
        .nii
        .nii.gz

    Also supports the non-standard segmentation filename
    present in BraTS20_Training_355 in the Kaggle dataset.
    """

    root = Path(root)

    if not root.exists():
        raise FileNotFoundError(
            f"BraTS 2020 root does not exist: {root}"
        )

    subjects: List[Path] = []

    for subject_dir in sorted(
        p for p in root.iterdir()
        if p.is_dir()
    ):
        subject_id = subject_dir.name

        try:
            for modality in MODALITIES:
                find_file(
                    subject_dir,
                    subject_id,
                    modality,
                )

            find_file(
                subject_dir,
                subject_id,
                LABEL_NAME,
            )

            subjects.append(subject_dir)

        except FileNotFoundError:
            continue

    if not subjects:
        raise RuntimeError(
            f"No valid BraTS 2020 subjects found under {root}."
        )

    return subjects


def load_nifti(path: Path):
    """Load a NIfTI file."""
    image = nib.load(str(path))
    data = image.get_fdata(dtype=np.float32)
    return image, data


def validate_subject(
    subject_dir: str | Path,
) -> Dict:
    """Validate one BraTS 2020 subject."""

    subject_dir = Path(subject_dir)
    subject_id = subject_dir.name

    paths = {
        modality: find_file(
            subject_dir,
            subject_id,
            modality,
        )
        for modality in MODALITIES
    }

    seg_path = find_file(
        subject_dir,
        subject_id,
        LABEL_NAME,
    )

    reference_img, reference = load_nifti(
        paths["t1"]
    )

    report = {
        "subject_id": subject_id,
        "shape": tuple(reference.shape),
        "affine": reference_img.affine.copy(),
        "spacing": tuple(
            reference_img.header.get_zooms()[:3]
        ),
        "modalities": {},
        "label_values": None,
    }

    for modality, path in paths.items():

        image, data = load_nifti(path)

        if data.shape != reference.shape:
            raise ValueError(
                f"{subject_id}: {modality} shape "
                f"{data.shape} != reference "
                f"{reference.shape}"
            )

        if not np.allclose(
            image.affine,
            reference_img.affine,
            atol=1e-4,
        ):
            raise ValueError(
                f"{subject_id}: {modality} affine "
                f"does not match T1"
            )

        report["modalities"][modality] = {
            "shape": tuple(data.shape),
            "min": float(np.nanmin(data)),
            "max": float(np.nanmax(data)),
            "mean": float(np.nanmean(data)),
            "std": float(np.nanstd(data)),
        }

    seg_img, seg = load_nifti(seg_path)

    if seg.shape != reference.shape:
        raise ValueError(
            f"{subject_id}: segmentation shape "
            f"{seg.shape} != reference "
            f"{reference.shape}"
        )

    if not np.allclose(
        seg_img.affine,
        reference_img.affine,
        atol=1e-4,
    ):
        raise ValueError(
            f"{subject_id}: segmentation affine "
            f"does not match T1"
        )

    unique = np.unique(
        seg.astype(np.int16)
    )

    allowed = np.array(
        [0, 1, 2, 4]
    )

    if not np.all(
        np.isin(unique, allowed)
    ):
        raise ValueError(
            f"{subject_id}: unexpected "
            f"segmentation labels "
            f"{unique.tolist()}"
        )

    report["label_values"] = unique.tolist()

    return report


def remap_labels(
    seg: np.ndarray,
) -> np.ndarray:
    """
    Convert original BraTS labels:

        0 -> 0 background
        1 -> 1 necrosis
        2 -> 2 edema
        4 -> 3 enhancing tumor

    This produces contiguous classes required
    by a 4-class CrossEntropyLoss.
    """

    seg = np.asarray(seg)

    output = np.zeros(
        seg.shape,
        dtype=np.int64,
    )

    output[seg == 1] = 1
    output[seg == 2] = 2
    output[seg == 4] = 3

    return output


def zscore_nonzero(
    volume: np.ndarray,
    eps: float = 1e-8,
) -> np.ndarray:
    """Z-score non-zero voxels while preserving background."""

    volume = np.asarray(
        volume,
        dtype=np.float32,
    )

    mask = volume != 0

    if not np.any(mask):
        return volume.copy()

    values = volume[mask]

    mean = float(values.mean())
    std = float(values.std())

    output = np.zeros_like(
        volume,
        dtype=np.float32,
    )

    if std < eps:
        output[mask] = (
            volume[mask] - mean
        )
    else:
        output[mask] = (
            volume[mask] - mean
        ) / std

    return output


class BraTS2020Dataset(Dataset):
    """
    BraTS 2020 multimodal dataset.

    Returns:

        image: [4, D, H, W]
        mask:  [D, H, W]

    Modalities:

        channel 0 = T1
        channel 1 = T1CE
        channel 2 = T2
        channel 3 = FLAIR

    Mask classes:

        0 = background
        1 = necrosis
        2 = edema
        3 = enhancing tumor
    """

    def __init__(
        self,
        root: str | Path,
        subjects: Optional[
            Sequence[Path]
        ] = None,
        transform=None,
        normalize: bool = True,
    ):
        self.root = Path(root)

        self.subjects = (
            list(subjects)
            if subjects is not None
            else discover_subjects(self.root)
        )

        self.transform = transform
        self.normalize = normalize

    def __len__(self):
        return len(self.subjects)

    def __getitem__(
        self,
        index: int,
    ):

        subject_dir = Path(
            self.subjects[index]
        )

        subject_id = subject_dir.name

        arrays = []
        reference_img = None

        # Load T1, T1CE, T2, FLAIR
        for modality in MODALITIES:

            path = find_file(
                subject_dir,
                subject_id,
                modality,
            )

            image, data = load_nifti(path)

            if reference_img is None:
                reference_img = image

            if data.shape != reference_img.shape:
                raise ValueError(
                    f"{subject_id}: modality "
                    f"{modality} shape mismatch"
                )

            if self.normalize:
                data = zscore_nonzero(data)

            arrays.append(data)

        # Load segmentation
        seg_path = find_file(
            subject_dir,
            subject_id,
            LABEL_NAME,
        )

        _, seg = load_nifti(seg_path)

        if seg.shape != arrays[0].shape:
            raise ValueError(
                f"{subject_id}: segmentation shape "
                f"{seg.shape} does not match image "
                f"{arrays[0].shape}"
            )

        # Convert 0,1,2,4 -> 0,1,2,3
        seg = remap_labels(seg)

        sample = {
            "image": torch.from_numpy(
                np.stack(
                    arrays,
                    axis=0,
                ).astype(np.float32)
            ),

            "mask": torch.from_numpy(
                seg.astype(np.int64)
            ),

            "subject_id": subject_id,

            "affine": reference_img.affine.copy(),

            "header": reference_img.header.copy(),
        }

        if self.transform is not None:
            sample = self.transform(sample)

        return sample


def validate_dataset(
    root: str | Path,
    limit: Optional[int] = None,
) -> List[Dict]:

    subjects = discover_subjects(root)

    if limit is not None:
        subjects = subjects[:limit]

    return [
        validate_subject(subject)
        for subject in subjects
    ]
'''

dataset_file.write_text(
    code,
    encoding="utf-8"
)

print("✅ brats2020_dataset.py replaced")
print(dataset_file)

✅ brats2020_dataset.py replaced
/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation/v2/data/brats2020_dataset.py


In [14]:
# Step 8 — Reload the updated dataset module
import importlib
import v2.data.brats2020_dataset as brats2020_dataset

importlib.reload(brats2020_dataset)

print("✅ Dataset module reloaded")

✅ Dataset module reloaded


In [15]:
# Step 9 — discover the subjects using the corrected loader.
subjects = brats2020_dataset.discover_subjects(DATA_ROOT)

print("Number of subjects:", len(subjects))
print("First subject:", subjects[0])
print("Last subject:", subjects[-1])

Number of subjects: 369
First subject: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001
Last subject: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_369


In [17]:
# Step 10 — Test normal subject + problematic subject 355
from v2.data.brats2020_dataset import BraTS2020Dataset

# Test first subject
dataset_test = BraTS2020Dataset(
    root=DATA_ROOT,
    subjects=[subjects[0]],
    normalize=True,
)

sample = dataset_test[0]

print("===== SUBJECT 001 =====")
print("Subject:", sample["subject_id"])
print("Image shape:", sample["image"].shape)
print("Mask shape:", sample["mask"].shape)
print("Image dtype:", sample["image"].dtype)
print("Mask dtype:", sample["mask"].dtype)
print("Mask classes:", torch.unique(sample["mask"]))


# Find subject 355
subject_355 = next(
    s for s in subjects
    if s.name == "BraTS20_Training_355"
)

# Test subject 355 specifically
dataset_355 = BraTS2020Dataset(
    root=DATA_ROOT,
    subjects=[subject_355],
    normalize=True,
)

sample_355 = dataset_355[0]

print("\n===== SUBJECT 355 =====")
print("Subject:", sample_355["subject_id"])
print("Image shape:", sample_355["image"].shape)
print("Mask shape:", sample_355["mask"].shape)
print("Image dtype:", sample_355["image"].dtype)
print("Mask dtype:", sample_355["mask"].dtype)
print("Mask classes:", torch.unique(sample_355["mask"]))

print("\n✅ DATASET LOADING TEST PASSED")

===== SUBJECT 001 =====
Subject: BraTS20_Training_001
Image shape: torch.Size([4, 240, 240, 155])
Mask shape: torch.Size([240, 240, 155])
Image dtype: torch.float32
Mask dtype: torch.int64
Mask classes: tensor([0, 1, 2, 3])

===== SUBJECT 355 =====
Subject: BraTS20_Training_355
Image shape: torch.Size([4, 240, 240, 155])
Mask shape: torch.Size([240, 240, 155])
Image dtype: torch.float32
Mask dtype: torch.int64
Mask classes: tensor([0, 1, 2, 3])

✅ DATASET LOADING TEST PASSED


In [18]:
# Step 11 — Test the patch sampler
# Step 11A — Check whether the file exists
from pathlib import Path

repo = Path("/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation")
patch_file = repo / "v2" / "data" / "patch_sampling.py"

print("Path:", patch_file)
print("Exists:", patch_file.exists())

if patch_file.exists():
    print("Size:", patch_file.stat().st_size, "bytes")


# Step 11B — Create patch_sampling.py
from pathlib import Path

patch_sampling_code = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import torch
from torch.utils.data import Dataset


@dataclass
class PatchSamplingConfig:
    """Configuration for 3D BraTS patch sampling."""

    patch_size: Tuple[int, int, int] = (64, 64, 64)

    # Probability of deliberately sampling a tumor-containing patch.
    tumor_probability: float = 0.8

    # Minimum number of tumor voxels required for a tumor patch.
    min_tumor_voxels: int = 1

    # Maximum number of attempts to find a tumor-containing patch.
    max_tumor_attempts: int = 20

    # Number of patches generated from each subject per epoch.
    patches_per_subject: int = 2


class BraTSPatchDataset(Dataset):
    """
    Patch-based wrapper around BraTS2020Dataset.

    Input dataset must return:
        image: [4, D, H, W]
        mask:  [D, H, W]

    This dataset returns:
        image: [4, pd, ph, pw]
        mask:  [pd, ph, pw]

    Sampling is tumor-aware: most patches are deliberately selected
    from regions containing tumor voxels.
    """

    def __init__(
        self,
        dataset,
        config: Optional[PatchSamplingConfig] = None,
    ):
        self.dataset = dataset
        self.config = config or PatchSamplingConfig()

        self.patch_size = tuple(self.config.patch_size)

        if len(self.patch_size) != 3:
            raise ValueError(
                f"patch_size must contain 3 values, got {self.patch_size}"
            )

        if any(size <= 0 for size in self.patch_size):
            raise ValueError(
                f"patch_size values must be positive, got {self.patch_size}"
            )

        if not 0.0 <= self.config.tumor_probability <= 1.0:
            raise ValueError(
                "tumor_probability must be between 0 and 1"
            )

        if self.config.patches_per_subject <= 0:
            raise ValueError(
                "patches_per_subject must be greater than 0"
            )

        # Cache valid subject indices. This prevents accidental
        # inclusion of malformed subjects in later training.
        self.subject_indices = list(range(len(self.dataset)))

    def __len__(self):
        return (
            len(self.subject_indices)
            * self.config.patches_per_subject
        )

    def _get_subject_index(self, index: int) -> int:
        return self.subject_indices[
            index // self.config.patches_per_subject
        ]

    @staticmethod
    def _validate_sample(sample: Dict):
        if "image" not in sample:
            raise KeyError("Dataset sample is missing 'image'")

        if "mask" not in sample:
            raise KeyError("Dataset sample is missing 'mask'")

        image = sample["image"]
        mask = sample["mask"]

        if not isinstance(image, torch.Tensor):
            image = torch.as_tensor(image)

        if not isinstance(mask, torch.Tensor):
            mask = torch.as_tensor(mask)

        if image.ndim != 4:
            raise ValueError(
                f"Expected image [C,D,H,W], got {tuple(image.shape)}"
            )

        if mask.ndim != 3:
            raise ValueError(
                f"Expected mask [D,H,W], got {tuple(mask.shape)}"
            )

        if tuple(image.shape[1:]) != tuple(mask.shape):
            raise ValueError(
                f"Image spatial shape {tuple(image.shape[1:])} "
                f"does not match mask shape {tuple(mask.shape)}"
            )

        return image.float(), mask.long()

    def _valid_start_range(
        self,
        spatial_shape: Tuple[int, int, int],
    ):
        starts = []

        for dimension, patch in zip(
            spatial_shape,
            self.patch_size,
        ):
            if dimension < patch:
                raise ValueError(
                    f"Volume dimension {dimension} is smaller than "
                    f"patch dimension {patch}"
                )

            starts.append(dimension - patch)

        return starts

    def _random_start(
        self,
        spatial_shape: Tuple[int, int, int],
    ) -> Tuple[int, int, int]:
        max_starts = self._valid_start_range(spatial_shape)

        return tuple(
            int(torch.randint(0, maximum + 1, (1,)).item())
            for maximum in max_starts
        )

    def _extract_patch(
        self,
        image: torch.Tensor,
        mask: torch.Tensor,
        start: Tuple[int, int, int],
    ):
        d, h, w = start
        pd, ph, pw = self.patch_size

        image_patch = image[
            :,
            d:d + pd,
            h:h + ph,
            w:w + pw,
        ]

        mask_patch = mask[
            d:d + pd,
            h:h + ph,
            w:w + pw,
        ]

        return image_patch.contiguous(), mask_patch.contiguous()

    def _find_tumor_start(
        self,
        image: torch.Tensor,
        mask: torch.Tensor,
    ):
        tumor = mask > 0

        if not torch.any(tumor):
            return None

        tumor_coordinates = torch.nonzero(
            tumor,
            as_tuple=False,
        )

        spatial_shape = tuple(mask.shape)

        for _ in range(self.config.max_tumor_attempts):
            # Pick an actual tumor voxel as the center/anchor.
            selected = tumor_coordinates[
                torch.randint(
                    0,
                    tumor_coordinates.shape[0],
                    (1,),
                ).item()
            ]

            center_d, center_h, center_w = (
                int(selected[0]),
                int(selected[1]),
                int(selected[2]),
            )

            starts = []

            for center, dimension, patch in zip(
                (center_d, center_h, center_w),
                spatial_shape,
                self.patch_size,
            ):
                maximum_start = dimension - patch

                lower = max(0, center - patch + 1)
                upper = min(center, maximum_start)

                if lower > upper:
                    lower = 0
                    upper = maximum_start

                start = int(
                    torch.randint(
                        lower,
                        upper + 1,
                        (1,),
                    ).item()
                )

                starts.append(start)

            start = tuple(starts)

            _, candidate_mask = self._extract_patch(
                image,
                mask,
                start,
            )

            tumor_voxels = int(
                torch.count_nonzero(candidate_mask > 0).item()
            )

            if tumor_voxels >= self.config.min_tumor_voxels:
                return start

        return None

    def __getitem__(self, index: int) -> Dict:
        subject_index = self._get_subject_index(index)

        sample = self.dataset[subject_index]

        image, mask = self._validate_sample(sample)

        spatial_shape = tuple(mask.shape)

        use_tumor_sampling = (
            torch.rand(()).item()
            < self.config.tumor_probability
        )

        start = None
        sampling_mode = "random"

        if use_tumor_sampling:
            start = self._find_tumor_start(
                image,
                mask,
            )

            if start is not None:
                sampling_mode = "tumor"

        if start is None:
            start = self._random_start(
                spatial_shape
            )

        image_patch, mask_patch = self._extract_patch(
            image,
            mask,
            start,
        )

        output = dict(sample)

        output["image"] = image_patch
        output["mask"] = mask_patch

        # Useful metadata for debugging and training statistics.
        output["patch_start"] = start
        output["sampling_mode"] = sampling_mode
        output["tumor_voxels"] = int(
            torch.count_nonzero(mask_patch > 0).item()
        )
        output["subject_index"] = subject_index

        return output


def build_patch_dataset(
    dataset,
    patch_size: Tuple[int, int, int] = (64, 64, 64),
    tumor_probability: float = 0.8,
    patches_per_subject: int = 2,
):
    """Convenience function for constructing the patch dataset."""

    config = PatchSamplingConfig(
        patch_size=patch_size,
        tumor_probability=tumor_probability,
        patches_per_subject=patches_per_subject,
    )

    return BraTSPatchDataset(
        dataset=dataset,
        config=config,
    )
'''

repo = Path(
    "/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation"
)

patch_file = repo / "v2" / "data" / "patch_sampling.py"

patch_file.parent.mkdir(parents=True, exist_ok=True)
patch_file.write_text(patch_sampling_code, encoding="utf-8")

print("✅ patch_sampling.py created")
print(patch_file)
print("Size:", patch_file.stat().st_size, "bytes")

# Step 11C — Reload and import the patch sampler

import importlib
from v2.data import patch_sampling

patch_sampling = importlib.reload(patch_sampling)

print("✅ patch_sampling imported successfully")
print("BraTSPatchDataset:", patch_sampling.BraTSPatchDataset)
print("PatchSamplingConfig:", patch_sampling.PatchSamplingConfig)

Path: /kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation/v2/data/patch_sampling.py
Exists: False
✅ patch_sampling.py created
/kaggle/working/3d-RCNN-using-Unet-for-brain-tumor-segmentation/v2/data/patch_sampling.py
Size: 8521 bytes
✅ patch_sampling imported successfully
BraTSPatchDataset: <class 'v2.data.patch_sampling.BraTSPatchDataset'>
PatchSamplingConfig: <class 'v2.data.patch_sampling.PatchSamplingConfig'>


In [2]:
# Step 12 — Test one training patch
import torch

from v2.data.brats2020_dataset import BraTS2020Dataset
from v2.data.patch_sampling import (
    BraTSPatchDataset,
    PatchSamplingConfig,
)

# Use one already-validated subject
base_dataset = BraTS2020Dataset(
    root=DATA_ROOT,
    subjects=[subjects[0]],
    normalize=True,
)

patch_config = PatchSamplingConfig(
    patch_size=(64, 64, 64),
    tumor_probability=0.8,
    patches_per_subject=1,
)

patch_dataset = BraTSPatchDataset(
    dataset=base_dataset,
    config=patch_config,
)

patch = patch_dataset[0]

print("===== PATCH TEST =====")
print("Subject:", patch["subject_id"])
print("Full subject image:", base_dataset[0]["image"].shape)
print("Patch image:", patch["image"].shape)
print("Patch mask:", patch["mask"].shape)
print("Image dtype:", patch["image"].dtype)
print("Mask dtype:", patch["mask"].dtype)
print("Mask classes:", torch.unique(patch["mask"]))
print("Patch start:", patch["patch_start"])
print("Tumor voxels:", patch["tumor_voxels"])
print("Sampling mode:", patch["sampling_mode"])

assert patch["image"].shape == torch.Size([4, 64, 64, 64])
assert patch["mask"].shape == torch.Size([64, 64, 64])
assert patch["image"].dtype == torch.float32
assert patch["mask"].dtype == torch.int64

print("✅ 64³ PATCH TEST PASSED")

ModuleNotFoundError: No module named 'v2'

In [1]:
# Step 13 — Create the real training/validation split

# Now we'll create the split before creating the DataLoaders.

# We have 369 subjects. We'll use:

# 295 training
# 74 validation
# No patient overlap
# Fixed random seed so the split is reproducible

import random
import numpy as np
import torch

from v2.data.brats2020_dataset import BraTS2020Dataset

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# Shuffle subjects
# ============================================================

all_subjects = list(subjects)

random.Random(SEED).shuffle(all_subjects)

# ============================================================
# Train / validation split
# ============================================================

NUM_TOTAL = len(all_subjects)
NUM_VAL = 74
NUM_TRAIN = NUM_TOTAL - NUM_VAL

train_subjects = all_subjects[:NUM_TRAIN]
val_subjects = all_subjects[NUM_TRAIN:]

print("===== DATA SPLIT =====")
print("Total subjects :", NUM_TOTAL)
print("Train subjects :", len(train_subjects))
print("Val subjects   :", len(val_subjects))

# ============================================================
# Verify no leakage
# ============================================================

train_ids = {p.name for p in train_subjects}
val_ids = {p.name for p in val_subjects}

overlap = train_ids.intersection(val_ids)

print("Train IDs:", len(train_ids))
print("Val IDs:", len(val_ids))
print("Overlap:", len(overlap))

assert NUM_TOTAL == 369
assert len(train_subjects) == 295
assert len(val_subjects) == 74
assert len(overlap) == 0

print("✅ NO PATIENT LEAKAGE")

print("\nFirst 5 training subjects:")
for subject in train_subjects[:5]:
    print(" ", subject.name)

print("\nFirst 5 validation subjects:")
for subject in val_subjects[:5]:
    print(" ", subject.name)

ModuleNotFoundError: No module named 'v2'

In [ ]:
# Step 14 — Create the training and validation patch datasets

# Now we'll connect the split we just created to the BraTSPatchDataset.
from v2.data.patch_sampling import (
    BraTSPatchDataset,
    PatchSamplingConfig,
)

# ============================================================
# Patch configuration
# ============================================================

patch_config = PatchSamplingConfig(
    patch_size=(64, 64, 64),
    patches_per_subject=2,
    tumor_probability=0.9,
)

# ============================================================
# Training patch dataset
# ============================================================

train_base_dataset = BraTS2020Dataset(
    root=DATA_ROOT,
    subjects=train_subjects,
    normalize=True,
)

train_patch_dataset = BraTSPatchDataset(
    dataset=train_base_dataset,
    config=patch_config,
)

# ============================================================
# Validation patch dataset
# ============================================================

val_base_dataset = BraTS2020Dataset(
    root=DATA_ROOT,
    subjects=val_subjects,
    normalize=True,
)

val_patch_dataset = BraTSPatchDataset(
    dataset=val_base_dataset,
    config=patch_config,
)

# ============================================================
# Summary
# ============================================================

print("===== PATCH DATASETS =====")
print("Training subjects  :", len(train_subjects))
print("Validation subjects:", len(val_subjects))

print("Training patches   :", len(train_patch_dataset))
print("Validation patches :", len(val_patch_dataset))

print("Patch size         :", patch_config.patch_size)
print("Patches/subject    :", patch_config.patches_per_subject)
print("Tumor probability  :", patch_config.tumor_probability)

assert len(train_patch_dataset) == 295 * 2
assert len(val_patch_dataset) == 74 * 2

print("✅ PATCH DATASETS CREATED")

In [ ]:
# Step 15 — Test 10 training patches
import torch

print("===== TRAINING PATCH TEST =====")

tumor_patches = 0
total_patches = 10

for i in range(total_patches):
    sample = train_patch_dataset[i]

    image = sample["image"]
    mask = sample["mask"]

    print(
        f"Patch {i + 1:02d} | "
        f"Subject: {sample['subject_id']} | "
        f"Image: {tuple(image.shape)} | "
        f"Mask: {tuple(mask.shape)} | "
        f"Tumor voxels: {sample['tumor_voxels']} | "
        f"Mode: {sample['sampling_mode']}"
    )

    assert image.shape == torch.Size([4, 64, 64, 64])
    assert mask.shape == torch.Size([64, 64, 64])
    assert image.dtype == torch.float32
    assert mask.dtype == torch.int64

    if sample["tumor_voxels"] > 0:
        tumor_patches += 1

print()
print("Tumor-containing patches:", f"{tumor_patches}/{total_patches}")
print(
    "Tumor patch rate:",
    f"{100 * tumor_patches / total_patches:.1f}%"
)

print("✅ TRAINING PATCH TEST PASSED")

In [ ]:
# ============================================================
# STEP 16: BASE DATASETS
# ============================================================
# Purpose:
#   1. Locate the BraTS 2020 dataset
#   2. Discover the actual subject folders
#   3. Create a reproducible 295/74 experiment split
#   4. Create train_dataset and val_dataset
#   5. Validate both datasets
#   6. Save the exact subject split for future recovery
# ============================================================

from pathlib import Path
import json
import random
import torch

from v2.data.brats2020_dataset import BraTS2020Dataset


# ============================================================
# 16.1 CONFIGURATION
# ============================================================

print("=" * 60)
print("===== STEP 16.1: CONFIGURATION =====")
print("=" * 60)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/"
    "brats20-dataset-training-validation"
)

TRAIN_ROOT = (
    DATASET_ROOT
    / "BraTS2020_TrainingData"
    / "MICCAI_BraTS2020_TrainingData"
)

VAL_ROOT = (
    DATASET_ROOT
    / "BraTS2020_ValidationData"
    / "MICCAI_BraTS2020_ValidationData"
)

# The experiment previously used 369 subjects:
# 295 training + 74 validation.
#
# IMPORTANT:
# This seed makes the split reproducible.
# The exact old split cannot be guaranteed unless its original
# subject list or seed is recovered.
SPLIT_SEED = 42

NUM_TRAIN_SUBJECTS = 295
NUM_VAL_SUBJECTS = 74

SPLIT_FILE = Path(
    "/kaggle/working/brats2020_experiment_split.json"
)

print("Dataset root :", DATASET_ROOT)
print("Training root:", TRAIN_ROOT)
print("Validation root:", VAL_ROOT)
print("Split seed   :", SPLIT_SEED)

assert DATASET_ROOT.exists(), (
    f"❌ Dataset root does not exist:\n{DATASET_ROOT}"
)

assert TRAIN_ROOT.exists(), (
    f"❌ Training root does not exist:\n{TRAIN_ROOT}"
)

assert VAL_ROOT.exists(), (
    f"❌ Validation root does not exist:\n{VAL_ROOT}"
)

print("✅ Dataset paths verified")


# ============================================================
# 16.2 DISCOVER RAW SUBJECTS
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.2: SUBJECT DISCOVERY =====")
print("=" * 60)

all_train_subjects = sorted(
    [
        path
        for path in TRAIN_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda p: p.name
)

all_val_subjects = sorted(
    [
        path
        for path in VAL_ROOT.iterdir()
        if path.is_dir()
    ],
    key=lambda p: p.name
)

print("Raw training subjects   :", len(all_train_subjects))
print("Raw validation subjects :", len(all_val_subjects))

assert len(all_train_subjects) == 369, (
    f"❌ Expected 369 raw training subjects, "
    f"found {len(all_train_subjects)}"
)

assert len(all_val_subjects) == 125, (
    f"❌ Expected 125 raw validation subjects, "
    f"found {len(all_val_subjects)}"
)

print("✅ Raw dataset structure verified")


# ============================================================
# 16.3 CREATE EXPERIMENT SPLIT
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.3: EXPERIMENT SPLIT =====")
print("=" * 60)

# Use only the 369 official BraTS training subjects.
#
# We intentionally do NOT mix the official BraTS validation
# subjects into our experiment split.

rng = random.Random(SPLIT_SEED)

shuffled_subjects = list(all_train_subjects)
rng.shuffle(shuffled_subjects)

train_subject_paths = shuffled_subjects[
    :NUM_TRAIN_SUBJECTS
]

val_subject_paths = shuffled_subjects[
    NUM_TRAIN_SUBJECTS:
    NUM_TRAIN_SUBJECTS + NUM_VAL_SUBJECTS
]

print("Split seed             :", SPLIT_SEED)
print("Experiment subjects    :", len(train_subject_paths) + len(val_subject_paths))
print("Training subjects      :", len(train_subject_paths))
print("Validation subjects    :", len(val_subject_paths))

assert len(train_subject_paths) == 295
assert len(val_subject_paths) == 74

assert len(
    set(train_subject_paths).intersection(
        set(val_subject_paths)
    )
) == 0

print("No train/validation overlap: ✅")
print("✅ Experiment split created")


# ============================================================
# 16.4 SAVE EXACT SPLIT
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.4: SAVE SPLIT =====")
print("=" * 60)

split_data = {
    "dataset": "BraTS2020",
    "split_seed": SPLIT_SEED,
    "raw_training_subjects": len(all_train_subjects),
    "raw_validation_subjects": len(all_val_subjects),
    "experiment_training_subjects": len(train_subject_paths),
    "experiment_validation_subjects": len(val_subject_paths),
    "train_subject_ids": [
        path.name for path in train_subject_paths
    ],
    "val_subject_ids": [
        path.name for path in val_subject_paths
    ],
}

with open(SPLIT_FILE, "w") as f:
    json.dump(split_data, f, indent=2)

assert SPLIT_FILE.exists()

print("Split file:", SPLIT_FILE)
print("Saved     : ✅")


# ============================================================
# 16.5 CREATE BASE DATASETS
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.5: CREATE DATASETS =====")
print("=" * 60)

train_dataset = BraTS2020Dataset(
    root=TRAIN_ROOT,
    subjects=train_subject_paths,
    normalize=True,
)

val_dataset = BraTS2020Dataset(
    root=TRAIN_ROOT,
    subjects=val_subject_paths,
    normalize=True,
)

print("Training dataset  :", len(train_dataset))
print("Validation dataset:", len(val_dataset))

assert len(train_dataset) == 295
assert len(val_dataset) == 74

print("Train dataset     : ✅")
print("Validation dataset: ✅")


# ============================================================
# 16.6 TRAINING SAMPLE VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.6: TRAINING SAMPLE =====")
print("=" * 60)

train_sample = train_dataset[0]

print("Keys:", list(train_sample.keys()))

print(
    "Image:",
    type(train_sample["image"]),
    train_sample["image"].shape,
    train_sample["image"].dtype
)

print(
    "Mask :",
    type(train_sample["mask"]),
    train_sample["mask"].shape,
    train_sample["mask"].dtype
)

assert isinstance(
    train_sample["image"],
    torch.Tensor
)

assert isinstance(
    train_sample["mask"],
    torch.Tensor
)

assert train_sample["image"].shape == (
    4, 240, 240, 155
)

assert train_sample["mask"].shape == (
    240, 240, 155
)

assert train_sample["image"].dtype == torch.float32
assert train_sample["mask"].dtype == torch.int64

assert torch.isfinite(
    train_sample["image"]
).all()

assert torch.isfinite(
    train_sample["mask"].float()
).all()

train_classes = torch.unique(
    train_sample["mask"]
)

print("Mask classes:", train_classes)

assert torch.all(
    (train_classes >= 0)
    & (train_classes <= 3)
)

print("Training sample: ✅")


# ============================================================
# 16.7 VALIDATION SAMPLE
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.7: VALIDATION SAMPLE =====")
print("=" * 60)

val_sample = val_dataset[0]

print("Keys:", list(val_sample.keys()))

print(
    "Image:",
    type(val_sample["image"]),
    val_sample["image"].shape,
    val_sample["image"].dtype
)

print(
    "Mask :",
    type(val_sample["mask"]),
    val_sample["mask"].shape,
    val_sample["mask"].dtype
)

assert isinstance(
    val_sample["image"],
    torch.Tensor
)

assert isinstance(
    val_sample["mask"],
    torch.Tensor
)

assert val_sample["image"].shape == (
    4, 240, 240, 155
)

assert val_sample["mask"].shape == (
    240, 240, 155
)

assert val_sample["image"].dtype == torch.float32
assert val_sample["mask"].dtype == torch.int64

assert torch.isfinite(
    val_sample["image"]
).all()

assert torch.isfinite(
    val_sample["mask"].float()
).all()

val_classes = torch.unique(
    val_sample["mask"]
)

print("Mask classes:", val_classes)

assert torch.all(
    (val_classes >= 0)
    & (val_classes <= 3)
)

print("Validation sample: ✅")


# ============================================================
# 16.8 FINAL STEP VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 16.8: FINAL VALIDATION =====")
print("=" * 60)

print("Dataset root exists       : ✅")
print("Training root exists      : ✅")
print("Validation root exists    : ✅")
print("Raw training subjects     : 369 ✅")
print("Raw validation subjects   : 125 ✅")
print("Experiment subjects       : 369 ✅")
print("Training subjects         : 295 ✅")
print("Validation subjects       : 74 ✅")
print("Train/validation overlap  : 0 ✅")
print("Train dataset             : ✅")
print("Validation dataset        : ✅")
print("Training sample           : ✅")
print("Validation sample         : ✅")
print("Tensor dtypes             : ✅")
print("Tensor shapes             : ✅")
print("Finite values             : ✅")
print("Mask classes              : ✅")
print("Split saved               : ✅")

print("\n" + "=" * 60)
print("✅ STEP 16 COMPLETE")
print("=" * 60)

print("train_dataset : READY")
print("val_dataset   : READY")
print("train subjects: 295")
print("val subjects  : 74")
print("split file    :", SPLIT_FILE)
print("=" * 60)

In [ ]:
# ============================================================
# STEP 17 FIX: REMOVE NIFTI HEADER FROM PATCH OUTPUT
# ============================================================

import os
import importlib
import inspect

print("=" * 60)
print("===== STEP 17 FIX: PATCH SAMPLER =====")
print("=" * 60)

PATCH_FILE = (
    "/kaggle/working/"
    "3d-RCNN-using-Unet-for-brain-tumor-segmentation/"
    "v2/data/patch_sampling.py"
)

if not os.path.exists(PATCH_FILE):
    raise FileNotFoundError(
        f"❌ patch_sampling.py not found:\n{PATCH_FILE}"
    )

print("Patch sampler file:")
print(PATCH_FILE)


# ------------------------------------------------------------
# Read source
# ------------------------------------------------------------

with open(PATCH_FILE, "r", encoding="utf-8") as f:
    source = f.read()


# ------------------------------------------------------------
# Replace unsafe output construction
# ------------------------------------------------------------

old_code = """        output = dict(sample)

        output["image"] = image_patch
        output["mask"] = mask_patch
"""

new_code = """        # Copy only DataLoader-safe metadata.
        # NIfTI headers are Python objects and must not be
        # returned through the PyTorch DataLoader.
        output = {
            key: value
            for key, value in sample.items()
            if key != "header"
        }

        output["image"] = image_patch
        output["mask"] = mask_patch
"""

if old_code not in source:
    print("\n⚠️ Expected code pattern was not found.")

    print("\nCurrent output construction:")
    start = source.find("output =")
    print(source[start:start + 500])

    raise RuntimeError(
        "❌ Could not safely patch patch_sampling.py. "
        "No changes were made."
    )


source = source.replace(
    old_code,
    new_code,
    1,
)


# ------------------------------------------------------------
# Write corrected source
# ------------------------------------------------------------

with open(PATCH_FILE, "w", encoding="utf-8") as f:
    f.write(source)

print("\n✅ patch_sampling.py updated")


# ------------------------------------------------------------
# Reload module
# ------------------------------------------------------------

import v2.data.patch_sampling as patch_sampling

importlib.reload(patch_sampling)

from v2.data.patch_sampling import (
    BraTSPatchDataset,
    PatchSamplingConfig,
)

print("✅ patch_sampling module reloaded")


# ------------------------------------------------------------
# Verify actual source
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("===== UPDATED __getitem__ =====")
print("=" * 60)

print(
    inspect.getsource(
        BraTSPatchDataset.__getitem__
    )
)


# ------------------------------------------------------------
# Verify required logic exists
# ------------------------------------------------------------

updated_source = inspect.getsource(
    BraTSPatchDataset.__getitem__
)

assert "key != \"header\"" in updated_source

print("\n" + "=" * 60)
print("✅ HEADER-REMOVAL LOGIC VERIFIED")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 17 FIX VERIFICATION: RECREATE PATCH DATASETS
# ============================================================

print("=" * 60)
print("===== STEP 17 FIX VERIFICATION =====")
print("=" * 60)

train_patch_dataset = BraTSPatchDataset(
    dataset=train_dataset,
    config=patch_config,
)

val_patch_dataset = BraTSPatchDataset(
    dataset=val_dataset,
    config=patch_config,
)

print("Training patches   :", len(train_patch_dataset))
print("Validation patches :", len(val_patch_dataset))

assert len(train_patch_dataset) == 590
assert len(val_patch_dataset) == 148

print("✅ Patch dataset counts verified")


# ------------------------------------------------------------
# Test one sample
# ------------------------------------------------------------

sample = train_patch_dataset[0]

print("\n===== PATCH SAMPLE KEYS =====")

for key in sample:
    print(" -", key)


# ------------------------------------------------------------
# Critical check
# ------------------------------------------------------------

assert "header" not in sample, (
    "❌ Nifti1Header is STILL present."
)

print("\n✅ NIfTI header successfully removed")

print("\n" + "=" * 60)
print("🎉 STEP 17 PATCH SAMPLER FIX SUCCESSFUL")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 17.6: DATALOADERS
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 17.6: DATALOADERS =====")
print("=" * 60)

from torch.utils.data import DataLoader

# ------------------------------------------------------------
# DataLoader configuration
# ------------------------------------------------------------

BATCH_SIZE = 1
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

print("Batch size   :", BATCH_SIZE)
print("Num workers  :", NUM_WORKERS)
print("Pin memory   :", PIN_MEMORY)

# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_patch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_patch_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("\nTraining batches  :", len(train_loader))
print("Validation batches:", len(val_loader))

assert len(train_loader) == 590
assert len(val_loader) == 148

print("✅ DataLoaders created")

In [ ]:
# ============================================================
# STEP 17.7: TRAINING DATALOADER TEST
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 17.7: TRAINING BATCH TEST =====")
print("=" * 60)

train_batch = next(iter(train_loader))

print("\nTraining batch keys:")

for key, value in train_batch.items():
    print(
        f"{key:16s} | "
        f"type={type(value)} | "
        f"shape={getattr(value, 'shape', 'N/A')}"
    )

train_images = train_batch["image"]
train_masks = train_batch["mask"]

print("\nImage dtype :", train_images.dtype)
print("Mask dtype  :", train_masks.dtype)

assert train_images.shape == (
    1, 4, 64, 64, 64
)

assert train_masks.shape == (
    1, 64, 64, 64
)

assert train_images.dtype == torch.float32
assert train_masks.dtype == torch.int64

assert torch.isfinite(train_images).all()
assert torch.isfinite(
    train_masks.float()
).all()

train_classes = torch.unique(train_masks)

print("Mask classes:", train_classes)

assert torch.all(
    (train_classes >= 0)
    & (train_classes <= 3)
)

assert "header" not in train_batch

print("\n✅ Training DataLoader works")

In [ ]:
# ============================================================
# STEP 17.8: VALIDATION DATALOADER TEST
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 17.8: VALIDATION BATCH TEST =====")
print("=" * 60)

val_batch = next(iter(val_loader))

val_images = val_batch["image"]
val_masks = val_batch["mask"]

print("Image shape :", val_images.shape)
print("Mask shape  :", val_masks.shape)

print("Image dtype :", val_images.dtype)
print("Mask dtype  :", val_masks.dtype)

assert val_images.shape == (
    1, 4, 64, 64, 64
)

assert val_masks.shape == (
    1, 64, 64, 64
)

assert val_images.dtype == torch.float32
assert val_masks.dtype == torch.int64

assert torch.isfinite(val_images).all()
assert torch.isfinite(
    val_masks.float()
).all()

val_classes = torch.unique(val_masks)

print("Mask classes:", val_classes)

assert torch.all(
    (val_classes >= 0)
    & (val_classes <= 3)
)

assert "header" not in val_batch

print("✅ Validation DataLoader works")

In [ ]:
# ============================================================
# STEP 17.9: FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("===== STEP 17.9: FINAL VALIDATION =====")
print("=" * 60)

print("Patch sampler        : ✅")
print("Patch configuration   : ✅")
print("Training subjects     : 295 ✅")
print("Validation subjects   : 74 ✅")
print("Training patches      : 590 ✅")
print("Validation patches    : 148 ✅")
print("Patch size            : 64 × 64 × 64 ✅")
print("Tumor probability     : 0.9 ✅")
print("NIfTI header removed  : ✅")
print("Training DataLoader   : ✅")
print("Validation DataLoader : ✅")
print("Image dtype           : float32 ✅")
print("Mask dtype            : int64 ✅")
print("Mask classes          : 0–3 ✅")
print("Finite values         : ✅")

print("\n" + "=" * 60)
print("✅ STEP 17 COMPLETE")
print("=" * 60)

print("train_patch_dataset : READY")
print("val_patch_dataset   : READY")
print("train_loader        : READY")
print("val_loader          : READY")

print("=" * 60)

In [ ]:
# ============================================================
# STEP 18 — COMPLETE MODEL + TRAINING PIPELINE TEST
# ============================================================

import torch

from v2.models.rcnn_unet3d import RCNNUNet3D
from v2.training.losses import DiceCrossEntropyLoss


# ============================================================
# 18.1 DEVICE
# ============================================================

print("===== STEP 18.1: DEVICE =====")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("Device  :", device)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

print("✅ Device ready")


# ============================================================
# 18.2 MODEL CREATION
# ============================================================

print()
print("===== STEP 18.2: MODEL =====")

model = RCNNUNet3D(
    in_channels=4,
    num_classes=4,
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Model type            :", type(model))
print("Model device          :", next(model.parameters()).device)
print("Total parameters      :", f"{total_params:,}")
print("Trainable parameters  :", f"{trainable_params:,}")

assert trainable_params == 7_215_988, (
    f"❌ Unexpected parameter count: {trainable_params:,}"
)

print("✅ Model created")


# ============================================================
# 18.3 LOSS
# ============================================================

print()
print("===== STEP 18.3: LOSS =====")

criterion = DiceCrossEntropyLoss(
    dice_weight=0.5,
    ce_weight=0.5,
)

print(criterion)
print("Dice weight      :", criterion.dice_weight)
print("Cross Entropy    :", criterion.ce_weight)

print("✅ Loss created")


# ============================================================
# 18.4 OPTIMIZER
# ============================================================

print()
print("===== STEP 18.4: OPTIMIZER =====")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5,
)

optimizer_params = sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
)

print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay :", optimizer.param_groups[0]["weight_decay"])
print("Optimizer parameters:", f"{optimizer_params:,}")

assert optimizer_params == trainable_params, (
    "❌ Optimizer does not contain all trainable parameters"
)

print("✅ Optimizer created")


# ============================================================
# 18.5 REAL TRAINING BATCH
# ============================================================

print()
print("===== STEP 18.5: TRAINING BATCH =====")

batch = next(iter(train_loader))

x = batch["image"].to(
    device,
    non_blocking=True,
)

y = batch["mask"].to(
    device,
    non_blocking=True,
)

print("Input :", x.shape)
print("Target:", y.shape)
print("Input device :", x.device)
print("Target device:", y.device)

assert x.shape == (
    1, 4, 64, 64, 64
)

assert y.shape == (
    1, 64, 64, 64
)

assert x.dtype == torch.float32
assert y.dtype == torch.int64

print("Mask classes:", torch.unique(y))

print("✅ Training batch ready")


# ============================================================
# 18.6 FORWARD PASS
# ============================================================

print()
print("===== STEP 18.6: FORWARD PASS =====")

model.train()

optimizer.zero_grad(set_to_none=True)

logits = model(x)

print("Output:", logits.shape)
print("Output dtype:", logits.dtype)

assert logits.shape == (
    1, 4, 64, 64, 64
), f"❌ Unexpected output shape: {logits.shape}"

assert torch.isfinite(logits).all(), (
    "❌ Model output contains NaN or Inf"
)

print("✅ Forward pass completed")


# ============================================================
# 18.7 LOSS CALCULATION
# ============================================================

print()
print("===== STEP 18.7: LOSS =====")

loss = criterion(
    logits,
    y,
)

print("Loss:", loss.item())

assert torch.isfinite(loss), (
    "❌ Loss is NaN or Inf"
)

print("✅ Loss calculated")


# ============================================================
# 18.8 BACKPROPAGATION
# ============================================================

print()
print("===== STEP 18.8: BACKPROPAGATION =====")

loss.backward()

gradient_count = 0
invalid_gradients = 0

for parameter in model.parameters():

    if parameter.grad is not None:

        gradient_count += 1

        if not torch.isfinite(
            parameter.grad
        ).all():

            invalid_gradients += 1

print("Parameters with gradients:", gradient_count)
print("Invalid gradients:", invalid_gradients)

assert gradient_count > 0, (
    "❌ No gradients generated"
)

assert invalid_gradients == 0, (
    "❌ NaN/Inf gradients detected"
)

print("✅ Backpropagation completed")
print("✅ Gradient check passed")


# ============================================================
# 18.9 OPTIMIZER STEP
# ============================================================

print()
print("===== STEP 18.9: OPTIMIZER STEP =====")

optimizer.step()

print("✅ Optimizer step completed")


# ============================================================
# 18.10 FINAL VALIDATION
# ============================================================

print()
print("===== STEP 18.10: FINAL VALIDATION =====")

print("Device             :", device)
print("Model parameters   :", f"{trainable_params:,}")
print("Input shape        :", tuple(x.shape))
print("Target shape       :", tuple(y.shape))
print("Output shape       :", tuple(logits.shape))
print("Loss               :", f"{loss.item():.6f}")
print("Gradient tensors   :", gradient_count)
print("Invalid gradients  :", invalid_gradients)

print()
print("============================================================")
print("✅ STEP 18 COMPLETE")
print("============================================================")
print("Device              : ✅")
print("Model               : ✅")
print("Loss                : ✅")
print("Optimizer           : ✅")
print("Forward pass        : ✅")
print("Loss calculation    : ✅")
print("Backpropagation     : ✅")
print("Gradient validation : ✅")
print("Optimizer step      : ✅")
print("============================================================")

In [ ]:
# ============================================================
# STEP 19 — TRAINING ENGINE SMOKE TEST
# ============================================================

import time
import torch


# ============================================================
# 19.1 CONFIGURATION
# ============================================================

SMOKE_TEST_BATCHES = 5

USE_AMP = (
    device.type == "cuda"
)

print("===== STEP 19.1: TRAINING CONFIG =====")

print("Device       :", device)
print("AMP enabled  :", USE_AMP)
print("Test batches :", SMOKE_TEST_BATCHES)

print("✅ Training configuration ready")


# ============================================================
# 19.2 AMP SCALER
# ============================================================

print()
print("===== STEP 19.2: AMP SCALER =====")

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("GradScaler:", scaler)
print("Enabled   :", scaler.is_enabled())

print("✅ AMP scaler ready")


# ============================================================
# 19.3 TRAINING FUNCTION
# ============================================================

def train_smoke_test(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    max_batches=5,
):
    """
    Small controlled training run used to validate
    the complete training engine before full training.
    """

    model.train()

    running_loss = 0.0
    processed_batches = 0

    start_time = time.time()

    for batch_idx, batch in enumerate(loader):

        if batch_idx >= max_batches:
            break

        # ----------------------------------------------------
        # Move data to GPU
        # ----------------------------------------------------

        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        # ----------------------------------------------------
        # Validate input
        # ----------------------------------------------------

        assert images.shape == (
            1, 4, 64, 64, 64
        ), (
            f"Unexpected image shape: {images.shape}"
        )

        assert masks.shape == (
            1, 64, 64, 64
        ), (
            f"Unexpected mask shape: {masks.shape}"
        )

        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward + loss
        # ----------------------------------------------------

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                masks,
            )

        # ----------------------------------------------------
        # Validate loss
        # ----------------------------------------------------

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite loss at batch "
                f"{batch_idx + 1}: {loss.item()}"
            )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        if USE_AMP:

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

        else:

            loss.backward()

            optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        loss_value = loss.detach().item()

        running_loss += loss_value
        processed_batches += 1

        print(
            f"Batch {batch_idx + 1}/{max_batches} "
            f"| Loss: {loss_value:.6f}"
        )

    elapsed = time.time() - start_time

    if processed_batches == 0:

        raise RuntimeError(
            "No batches were processed."
        )

    average_loss = (
        running_loss / processed_batches
    )

    return {
        "loss": average_loss,
        "batches": processed_batches,
        "time": elapsed,
    }


# ============================================================
# 19.4 RUN SMOKE TEST
# ============================================================

print()
print("===== STEP 19.3: TRAINING SMOKE TEST =====")

result = train_smoke_test(
    model=model,
    loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    max_batches=SMOKE_TEST_BATCHES,
)


# ============================================================
# 19.5 RESULTS
# ============================================================

print()
print("===== STEP 19.4: RESULTS =====")

print(
    "Batches processed:",
    result["batches"],
)

print(
    "Average loss     :",
    f"{result['loss']:.6f}",
)

print(
    "Time             :",
    f"{result['time']:.2f} seconds",
)

assert result["batches"] == SMOKE_TEST_BATCHES

assert torch.isfinite(
    torch.tensor(result["loss"])
)

print()
print("============================================================")
print("✅ STEP 19 COMPLETE")
print("============================================================")
print("Training batches  : ✅")
print("Forward pass      : ✅")
print("Loss calculation  : ✅")
print("Backpropagation   : ✅")
print("Optimizer updates : ✅")
print("AMP               :", "✅" if USE_AMP else "Disabled")
print("Finite loss       : ✅")
print("============================================================")

In [ ]:
# ============================================================
# STEP 20 — VALIDATION ENGINE SMOKE TEST
# ============================================================

import time
import torch


# ============================================================
# 20.1 CONFIGURATION
# ============================================================

VALIDATION_SMOKE_BATCHES = 5

print("===== STEP 20.1: VALIDATION CONFIG =====")

print("Device              :", device)
print("Validation batches  :", VALIDATION_SMOKE_BATCHES)
print("Classes             :", 4)
print("Background class    : 0")
print("Tumor classes       : 1, 2, 3")

print("✅ Validation configuration ready")


# ============================================================
# 20.2 DICE FUNCTION
# ============================================================

def dice_per_class(
    logits,
    target,
    num_classes=4,
    include_background=False,
    epsilon=1e-6,
):
    """
    Calculate Dice score independently for each class.

    logits:
        [B, C, D, H, W]

    target:
        [B, D, H, W]
    """

    prediction = torch.argmax(
        logits,
        dim=1,
    )

    dice_scores = []

    start_class = (
        0 if include_background else 1
    )

    for class_id in range(
        start_class,
        num_classes,
    ):

        pred_class = (
            prediction == class_id
        )

        target_class = (
            target == class_id
        )

        intersection = (
            pred_class & target_class
        ).sum().float()

        pred_volume = (
            pred_class.sum().float()
        )

        target_volume = (
            target_class.sum().float()
        )

        denominator = (
            pred_volume
            + target_volume
        )

        # ----------------------------------------------------
        # Both prediction and target contain no voxels
        # for this class.
        # ----------------------------------------------------

        if denominator == 0:

            dice = torch.tensor(
                1.0,
                device=logits.device,
            )

        else:

            dice = (
                2.0 * intersection
                + epsilon
            ) / (
                denominator
                + epsilon
            )

        dice_scores.append(dice)

    return torch.stack(dice_scores)


# ============================================================
# 20.3 VALIDATION FUNCTION
# ============================================================

def validate_smoke_test(
    model,
    loader,
    criterion,
    device,
    max_batches=5,
):
    """
    Controlled validation run.

    No gradients are calculated.
    """

    model.eval()

    running_loss = 0.0

    dice_sum = torch.zeros(
        3,
        device=device,
    )

    processed_batches = 0

    start_time = time.time()

    with torch.no_grad():

        for batch_idx, batch in enumerate(loader):

            if batch_idx >= max_batches:
                break

            # ------------------------------------------------
            # Move data to device
            # ------------------------------------------------

            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                device,
                non_blocking=True,
            )

            # ------------------------------------------------
            # Validate shapes
            # ------------------------------------------------

            assert images.shape == (
                1, 4, 64, 64, 64
            ), (
                f"Unexpected image shape: "
                f"{images.shape}"
            )

            assert masks.shape == (
                1, 64, 64, 64
            ), (
                f"Unexpected mask shape: "
                f"{masks.shape}"
            )

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            outputs = model(images)

            assert outputs.shape == (
                1, 4, 64, 64, 64
            ), (
                f"Unexpected output shape: "
                f"{outputs.shape}"
            )

            # ------------------------------------------------
            # Loss
            # ------------------------------------------------

            loss = criterion(
                outputs,
                masks,
            )

            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite validation loss "
                    f"at batch {batch_idx + 1}: "
                    f"{loss.item()}"
                )

            # ------------------------------------------------
            # Dice
            # ------------------------------------------------

            batch_dice = dice_per_class(
                logits=outputs,
                target=masks,
                num_classes=4,
                include_background=False,
            )

            if not torch.isfinite(
                batch_dice
            ).all():

                raise RuntimeError(
                    f"Non-finite Dice at "
                    f"batch {batch_idx + 1}"
                )

            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            running_loss += loss.item()

            dice_sum += batch_dice

            processed_batches += 1

            print(
                f"Batch {batch_idx + 1}/"
                f"{max_batches} "
                f"| Loss: {loss.item():.6f} "
                f"| Dice WT: {batch_dice[0].item():.4f} "
                f"| Dice TC: {batch_dice[1].item():.4f} "
                f"| Dice ET: {batch_dice[2].item():.4f}"
            )

    if processed_batches == 0:

        raise RuntimeError(
            "No validation batches were processed."
        )

    average_loss = (
        running_loss / processed_batches
    )

    mean_dice = (
        dice_sum / processed_batches
    )

    elapsed = time.time() - start_time

    return {
        "loss": average_loss,
        "dice_wt": mean_dice[0].item(),
        "dice_tc": mean_dice[1].item(),
        "dice_et": mean_dice[2].item(),
        "dice_mean": mean_dice.mean().item(),
        "batches": processed_batches,
        "time": elapsed,
    }


# ============================================================
# 20.4 RUN VALIDATION SMOKE TEST
# ============================================================

print()
print("===== STEP 20.2: VALIDATION SMOKE TEST =====")

validation_result = validate_smoke_test(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    max_batches=VALIDATION_SMOKE_BATCHES,
)


# ============================================================
# 20.5 RESULTS
# ============================================================

print()
print("===== STEP 20.3: VALIDATION RESULTS =====")

print(
    "Batches processed:",
    validation_result["batches"],
)

print(
    "Average loss     :",
    f"{validation_result['loss']:.6f}",
)

print(
    "Dice WT          :",
    f"{validation_result['dice_wt']:.6f}",
)

print(
    "Dice TC          :",
    f"{validation_result['dice_tc']:.6f}",
)

print(
    "Dice ET          :",
    f"{validation_result['dice_et']:.6f}",
)

print(
    "Mean Dice        :",
    f"{validation_result['dice_mean']:.6f}",
)

print(
    "Time             :",
    f"{validation_result['time']:.2f} seconds",
)


# ============================================================
# 20.6 FINAL VALIDATION
# ============================================================

assert (
    validation_result["batches"]
    == VALIDATION_SMOKE_BATCHES
)

assert torch.isfinite(
    torch.tensor(validation_result["loss"])
)

assert torch.isfinite(
    torch.tensor(validation_result["dice_mean"])
)

assert 0.0 <= validation_result["dice_mean"] <= 1.0

print()
print("============================================================")
print("✅ STEP 20 COMPLETE")
print("============================================================")
print("Validation batches : ✅")
print("Forward pass       : ✅")
print("Validation loss    : ✅")
print("Dice WT            : ✅")
print("Dice TC            : ✅")
print("Dice ET            : ✅")
print("Mean Dice          : ✅")
print("Finite metrics     : ✅")
print("No gradients       : ✅")
print("============================================================")

In [ ]:
# ============================================================
# STEP 21A — CORRECT BraTS WT / TC / ET METRICS
# ============================================================

import torch


# ============================================================
# 21A.1 BraTS REGION DEFINITIONS
# ============================================================

print("===== STEP 21A.1: BraTS REGION DEFINITIONS =====")

print("WT = labels 1 + 2 + 3")
print("TC = labels 1 + 3")
print("ET = label 3")

print("✅ BraTS region definitions loaded")


# ============================================================
# 21A.2 REGION DICE FUNCTION
# ============================================================

def brats_region_dice(
    logits,
    target,
    epsilon=1e-6,
):
    """
    Calculate standard BraTS regional Dice scores.

    Model labels:
        0 = background
        1 = tumor label 1
        2 = tumor label 2
        3 = tumor label 3

    BraTS regions:
        WT = labels 1, 2, 3
        TC = labels 1, 3
        ET = label 3

    Parameters
    ----------
    logits : torch.Tensor
        Shape [B, 4, D, H, W]

    target : torch.Tensor
        Shape [B, D, H, W]

    Returns
    -------
    torch.Tensor
        [WT, TC, ET]
    """

    prediction = torch.argmax(
        logits,
        dim=1,
    )

    # --------------------------------------------------------
    # Whole Tumor
    # --------------------------------------------------------

    pred_wt = prediction > 0
    target_wt = target > 0

    # --------------------------------------------------------
    # Tumor Core
    # --------------------------------------------------------

    pred_tc = (
        (prediction == 1)
        | (prediction == 3)
    )

    target_tc = (
        (target == 1)
        | (target == 3)
    )

    # --------------------------------------------------------
    # Enhancing Tumor
    # --------------------------------------------------------

    pred_et = prediction == 3
    target_et = target == 3

    regions = [
        (pred_wt, target_wt),
        (pred_tc, target_tc),
        (pred_et, target_et),
    ]

    scores = []

    for pred_region, target_region in regions:

        intersection = (
            pred_region & target_region
        ).sum().float()

        pred_volume = (
            pred_region.sum().float()
        )

        target_volume = (
            target_region.sum().float()
        )

        denominator = (
            pred_volume + target_volume
        )

        # ----------------------------------------------------
        # Both masks empty
        # ----------------------------------------------------

        if denominator.item() == 0:

            dice = torch.tensor(
                1.0,
                device=logits.device,
            )

        else:

            dice = (
                2.0 * intersection + epsilon
            ) / (
                denominator + epsilon
            )

        scores.append(dice)

    return torch.stack(scores)


# ============================================================
# 21A.3 TEST THE METRIC ON ONE BATCH
# ============================================================

print()
print("===== STEP 21A.2: SINGLE BATCH TEST =====")

model.eval()

test_batch = next(iter(val_loader))

test_images = test_batch["image"].to(
    device,
    non_blocking=True,
)

test_masks = test_batch["mask"].to(
    device,
    non_blocking=True,
)

with torch.no_grad():

    test_logits = model(
        test_images
    )

test_scores = brats_region_dice(
    test_logits,
    test_masks,
)

print("WT Dice:", f"{test_scores[0].item():.6f}")
print("TC Dice:", f"{test_scores[1].item():.6f}")
print("ET Dice:", f"{test_scores[2].item():.6f}")

assert test_scores.shape == (
    3,
)

assert torch.isfinite(
    test_scores
).all()

assert torch.all(
    test_scores >= 0
)

assert torch.all(
    test_scores <= 1
)

print("✅ Single-batch metric test passed")


# ============================================================
# 21A.4 FIVE-BATCH METRIC TEST
# ============================================================

print()
print("===== STEP 21A.3: FIVE-BATCH METRIC TEST =====")

NUM_TEST_BATCHES = 5

dice_sum = torch.zeros(
    3,
    device=device,
)

processed = 0

with torch.no_grad():

    for batch_idx, batch in enumerate(
        val_loader
    ):

        if batch_idx >= NUM_TEST_BATCHES:
            break

        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        logits = model(images)

        scores = brats_region_dice(
            logits,
            masks,
        )

        dice_sum += scores

        processed += 1

        print(
            f"Batch {batch_idx + 1}/"
            f"{NUM_TEST_BATCHES} "
            f"| WT: {scores[0].item():.4f} "
            f"| TC: {scores[1].item():.4f} "
            f"| ET: {scores[2].item():.4f}"
        )


# ============================================================
# 21A.5 FINAL METRICS
# ============================================================

mean_scores = (
    dice_sum / processed
)

mean_dice = mean_scores.mean()

print()
print("===== STEP 21A.4: METRIC RESULTS =====")

print(
    "Batches processed:",
    processed,
)

print(
    "WT Dice:",
    f"{mean_scores[0].item():.6f}",
)

print(
    "TC Dice:",
    f"{mean_scores[1].item():.6f}",
)

print(
    "ET Dice:",
    f"{mean_scores[2].item():.6f}",
)

print(
    "Mean Dice:",
    f"{mean_dice.item():.6f}",
)


# ============================================================
# 21A.6 FINAL CHECKS
# ============================================================

assert processed == NUM_TEST_BATCHES

assert torch.isfinite(
    mean_scores
).all()

assert torch.isfinite(
    mean_dice
)

assert torch.all(
    mean_scores >= 0
)

assert torch.all(
    mean_scores <= 1
)

print()
print("============================================================")
print("✅ STEP 21A COMPLETE")
print("============================================================")
print("BraTS WT metric : ✅")
print("BraTS TC metric : ✅")
print("BraTS ET metric : ✅")
print("Five-batch test : ✅")
print("Finite values   : ✅")
print("Range [0,1]     : ✅")
print("============================================================")

In [ ]:
# ============================================================
# STEP 21B — TRAINING + VALIDATION EPOCH ENGINE SMOKE TEST
# ============================================================

import time
import math
import torch


# ============================================================
# 21B.1 CONFIGURATION
# ============================================================

MAX_TRAIN_BATCHES = 10
MAX_VAL_BATCHES = 5

print("===== STEP 21B.1: EPOCH CONFIGURATION =====")

print("Device            :", device)
print("AMP enabled       :", True)
print("Train test batches:", MAX_TRAIN_BATCHES)
print("Val test batches  :", MAX_VAL_BATCHES)
print("Total train patches:", len(train_patch_dataset))
print("Total val patches  :", len(val_patch_dataset))

print("✅ Epoch configuration ready")


# ============================================================
# 21B.2 AMP SCALER
# ============================================================

scaler_21b = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

print()
print("===== STEP 21B.2: AMP SCALER =====")

print("Scaler enabled:", scaler_21b.is_enabled())

print("✅ AMP scaler ready")


# ============================================================
# 21B.3 TRAINING EPOCH
# ============================================================

def train_one_epoch_21b(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    max_batches=None,
):
    """
    Run one training epoch or a limited smoke-test epoch.
    """

    model.train()

    running_loss = 0.0
    processed_batches = 0

    start_time = time.time()

    optimizer.zero_grad(
        set_to_none=True
    )

    for batch_idx, batch in enumerate(loader):

        if (
            max_batches is not None
            and batch_idx >= max_batches
        ):
            break

        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = batch["mask"].to(
            device,
            non_blocking=True,
        )

        # ----------------------------------------------------
        # Input validation
        # ----------------------------------------------------

        assert images.shape == (
            1, 4, 64, 64, 64
        ), (
            f"Unexpected training image shape: "
            f"{images.shape}"
        )

        assert masks.shape == (
            1, 64, 64, 64
        ), (
            f"Unexpected training mask shape: "
            f"{masks.shape}"
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # AMP forward + loss
        # ----------------------------------------------------

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                masks,
            )

        # ----------------------------------------------------
        # Loss validation
        # ----------------------------------------------------

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite training loss "
                f"at batch {batch_idx + 1}: "
                f"{loss.item()}"
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        running_loss += loss.item()

        processed_batches += 1

        print(
            f"Train batch "
            f"{batch_idx + 1}/"
            f"{max_batches if max_batches else len(loader)} "
            f"| Loss: {loss.item():.6f}"
        )

    if processed_batches == 0:

        raise RuntimeError(
            "No training batches were processed."
        )

    average_loss = (
        running_loss / processed_batches
    )

    elapsed = time.time() - start_time

    return {
        "loss": average_loss,
        "batches": processed_batches,
        "time": elapsed,
    }


# ============================================================
# 21B.4 VALIDATION EPOCH
# ============================================================

def validate_one_epoch_21b(
    model,
    loader,
    criterion,
    device,
    max_batches=None,
):
    """
    Run one validation epoch.

    Calculates:
        validation loss
        WT Dice
        TC Dice
        ET Dice
        mean Dice
    """

    model.eval()

    running_loss = 0.0

    dice_sum = torch.zeros(
        3,
        device=device,
    )

    processed_batches = 0

    start_time = time.time()

    with torch.no_grad():

        for batch_idx, batch in enumerate(loader):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            masks = batch["mask"].to(
                device,
                non_blocking=True,
            )

            assert images.shape == (
                1, 4, 64, 64, 64
            ), (
                f"Unexpected validation image shape: "
                f"{images.shape}"
            )

            assert masks.shape == (
                1, 64, 64, 64
            ), (
                f"Unexpected validation mask shape: "
                f"{masks.shape}"
            )

            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda"),
            ):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    masks,
                )

            # ------------------------------------------------
            # Loss validation
            # ------------------------------------------------

            if not torch.isfinite(loss):

                raise RuntimeError(
                    f"Non-finite validation loss "
                    f"at batch {batch_idx + 1}: "
                    f"{loss.item()}"
                )

            # ------------------------------------------------
            # BraTS WT / TC / ET Dice
            # ------------------------------------------------

            scores = brats_region_dice(
                outputs,
                masks,
            )

            if not torch.isfinite(scores).all():

                raise RuntimeError(
                    f"Non-finite Dice at "
                    f"validation batch "
                    f"{batch_idx + 1}"
                )

            running_loss += loss.item()

            dice_sum += scores

            processed_batches += 1

            print(
                f"Val batch "
                f"{batch_idx + 1}/"
                f"{max_batches if max_batches else len(loader)} "
                f"| Loss: {loss.item():.6f} "
                f"| WT: {scores[0].item():.4f} "
                f"| TC: {scores[1].item():.4f} "
                f"| ET: {scores[2].item():.4f}"
            )

    if processed_batches == 0:

        raise RuntimeError(
            "No validation batches were processed."
        )

    average_loss = (
        running_loss / processed_batches
    )

    mean_scores = (
        dice_sum / processed_batches
    )

    mean_dice = mean_scores.mean()

    elapsed = time.time() - start_time

    return {
        "loss": average_loss,
        "dice_wt": mean_scores[0].item(),
        "dice_tc": mean_scores[1].item(),
        "dice_et": mean_scores[2].item(),
        "dice_mean": mean_dice.item(),
        "batches": processed_batches,
        "time": elapsed,
    }


# ============================================================
# 21B.5 RUN TRAINING SMOKE EPOCH
# ============================================================

print()
print("===== STEP 21B.3: TRAINING EPOCH SMOKE TEST =====")

epoch_start = time.time()

train_result_21b = train_one_epoch_21b(
    model=model,
    loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    scaler=scaler_21b,
    device=device,
    max_batches=MAX_TRAIN_BATCHES,
)

print()
print("===== STEP 21B.4: TRAINING RESULTS =====")

print(
    "Training batches:",
    train_result_21b["batches"],
)

print(
    "Training loss   :",
    f"{train_result_21b['loss']:.6f}",
)

print(
    "Training time   :",
    f"{train_result_21b['time']:.2f} seconds",
)


# ============================================================
# 21B.6 RUN VALIDATION SMOKE EPOCH
# ============================================================

print()
print("===== STEP 21B.5: VALIDATION EPOCH SMOKE TEST =====")

val_result_21b = validate_one_epoch_21b(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    max_batches=MAX_VAL_BATCHES,
)

print()
print("===== STEP 21B.6: VALIDATION RESULTS =====")

print(
    "Validation batches:",
    val_result_21b["batches"],
)

print(
    "Validation loss   :",
    f"{val_result_21b['loss']:.6f}",
)

print(
    "WT Dice           :",
    f"{val_result_21b['dice_wt']:.6f}",
)

print(
    "TC Dice           :",
    f"{val_result_21b['dice_tc']:.6f}",
)

print(
    "ET Dice           :",
    f"{val_result_21b['dice_et']:.6f}",
)

print(
    "Mean Dice         :",
    f"{val_result_21b['dice_mean']:.6f}",
)

print(
    "Validation time   :",
    f"{val_result_21b['time']:.2f} seconds",
)


# ============================================================
# 21B.7 FINAL CHECKS
# ============================================================

total_time = time.time() - epoch_start

print()
print("===== STEP 21B.7: FINAL ENGINE CHECK =====")

assert (
    train_result_21b["batches"]
    == MAX_TRAIN_BATCHES
)

assert (
    val_result_21b["batches"]
    == MAX_VAL_BATCHES
)

assert math.isfinite(
    train_result_21b["loss"]
)

assert math.isfinite(
    val_result_21b["loss"]
)

assert math.isfinite(
    val_result_21b["dice_wt"]
)

assert math.isfinite(
    val_result_21b["dice_tc"]
)

assert math.isfinite(
    val_result_21b["dice_et"]
)

assert math.isfinite(
    val_result_21b["dice_mean"]
)

assert 0.0 <= val_result_21b["dice_wt"] <= 1.0
assert 0.0 <= val_result_21b["dice_tc"] <= 1.0
assert 0.0 <= val_result_21b["dice_et"] <= 1.0
assert 0.0 <= val_result_21b["dice_mean"] <= 1.0


# ============================================================
# 21B.8 SUMMARY
# ============================================================

print()
print("============================================================")
print("✅ STEP 21B COMPLETE")
print("============================================================")
print("Training engine       : ✅")
print("Training batches      : ✅")
print("AMP training          : ✅")
print("Optimizer updates     : ✅")
print("Validation engine     : ✅")
print("Validation batches    : ✅")
print("Validation loss       : ✅")
print("BraTS WT Dice         : ✅")
print("BraTS TC Dice         : ✅")
print("BraTS ET Dice         : ✅")
print("Mean Dice             : ✅")
print("Finite values         : ✅")
print("Total test time       :", f"{total_time:.2f} seconds")
print("============================================================")

In [ ]:
# ============================================================
# STEP 22 — FRESH TRAINING INITIALIZATION
# ============================================================

import os
import torch


# ============================================================
# 22.1 PROJECT + TRAINING CONFIGURATION
# ============================================================

print("===== STEP 22.1: TRAINING CONFIGURATION =====")

PROJECT_ROOT = (
    "/kaggle/working/"
    "3d-RCNN-using-Unet-for-brain-tumor-segmentation"
)

NUM_EPOCHS = 50

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

NUM_CLASSES = 4
IN_CHANNELS = 4

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print("Project root    :", PROJECT_ROOT)
print("Epochs          :", NUM_EPOCHS)
print("Learning rate   :", LEARNING_RATE)
print("Weight decay    :", WEIGHT_DECAY)
print("Input channels  :", IN_CHANNELS)
print("Output classes  :", NUM_CLASSES)
print("Checkpoint dir  :", CHECKPOINT_DIR)

assert os.path.isdir(PROJECT_ROOT)
assert os.path.isdir(CHECKPOINT_DIR)

print("✅ Training configuration ready")


# ============================================================
# 22.2 FRESH MODEL
# ============================================================

print()
print("===== STEP 22.2: FRESH MODEL =====")

model = RCNNUNet3D(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
)

model = model.to(device)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Model type           :", type(model))
print(
    "Model device         :",
    next(model.parameters()).device
)
print(
    "Total parameters     :",
    f"{total_parameters:,}"
)
print(
    "Trainable parameters :",
    f"{trainable_parameters:,}"
)

assert total_parameters == 7_215_988
assert trainable_parameters == 7_215_988

print("✅ Fresh model created")


# ============================================================
# 22.3 FRESH LOSS
# ============================================================

print()
print("===== STEP 22.3: LOSS =====")

criterion = DiceCrossEntropyLoss(
    dice_weight=0.5,
    ce_weight=0.5,
)

print("Criterion:")
print(criterion)

print("Dice weight :", 0.5)
print("CE weight   :", 0.5)

print("✅ Fresh loss created")


# ============================================================
# 22.4 FRESH OPTIMIZER
# ============================================================

print()
print("===== STEP 22.4: OPTIMIZER =====")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

optimizer_parameter_count = sum(
    p.numel()
    for group in optimizer.param_groups
    for p in group["params"]
)

print("Optimizer:", optimizer)
print("Learning rate:", LEARNING_RATE)
print("Weight decay :", WEIGHT_DECAY)
print(
    "Optimizer parameters:",
    f"{optimizer_parameter_count:,}"
)

assert (
    optimizer_parameter_count
    == trainable_parameters
)

print("✅ Fresh optimizer created")


# ============================================================
# 22.5 LEARNING RATE SCHEDULER
# ============================================================

print()
print("===== STEP 22.5: LR SCHEDULER =====")

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

print("Scheduler:", scheduler)
print("Mode     : max")
print("Factor   : 0.5")
print("Patience : 3")
print("Min LR   : 1e-6")

print("✅ Scheduler created")


# ============================================================
# 22.6 FRESH AMP SCALER
# ============================================================

print()
print("===== STEP 22.6: AMP SCALER =====")

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

print("Scaler type:", type(scaler))
print("AMP enabled:", scaler.is_enabled())

assert (
    scaler.is_enabled()
    == (device.type == "cuda")
)

print("✅ Fresh AMP scaler created")


# ============================================================
# 22.7 TRAINING HISTORY
# ============================================================

print()
print("===== STEP 22.7: TRAINING HISTORY =====")

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "wt_dice": [],
    "tc_dice": [],
    "et_dice": [],
    "mean_dice": [],
    "learning_rate": [],
    "epoch_time": [],
}

print("History fields:")

for key in history:
    print(" ", key)

assert len(history) == 9

print("✅ Training history initialized")


# ============================================================
# 22.8 BEST MODEL TRACKING
# ============================================================

print()
print("===== STEP 22.8: BEST MODEL TRACKING =====")

best_mean_dice = -1.0
best_epoch = 0

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pth"
)

last_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "last_model.pth"
)

print("Best mean Dice :", best_mean_dice)
print("Best epoch     :", best_epoch)
print("Best checkpoint:", checkpoint_path)
print("Last checkpoint:", last_checkpoint_path)

print("✅ Checkpoint tracking initialized")


# ============================================================
# 22.9 EXPERIMENT STATE
# ============================================================

print()
print("===== STEP 22.9: EXPERIMENT STATE =====")

experiment_state = {
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_classes": NUM_CLASSES,
    "in_channels": IN_CHANNELS,
    "patch_size": (64, 64, 64),
    "train_subjects": len(
        train_patch_dataset.subject_indices
    ),
    "val_subjects": len(
        val_patch_dataset.subject_indices
    ),
    "train_patches": len(
        train_patch_dataset
    ),
    "val_patches": len(
        val_patch_dataset
    ),
    "amp_enabled": scaler.is_enabled(),
}

for key, value in experiment_state.items():
    print(f"{key:20s}: {value}")

print("✅ Experiment state ready")


# ============================================================
# 22.10 FINAL VALIDATION
# ============================================================

print()
print("===== STEP 22.10: FINAL VALIDATION =====")

assert next(
    model.parameters()
).device.type == device.type

assert (
    total_parameters
    == 7_215_988
)

assert (
    trainable_parameters
    == 7_215_988
)

assert (
    optimizer_parameter_count
    == trainable_parameters
)

assert len(history) == 9

assert best_mean_dice == -1.0
assert best_epoch == 0

assert os.path.isdir(
    CHECKPOINT_DIR
)

assert experiment_state[
    "train_subjects"
] == 295

assert experiment_state[
    "val_subjects"
] == 74

assert experiment_state[
    "train_patches"
] == 590

assert experiment_state[
    "val_patches"
] == 148

print("Model state       : ✅")
print("Optimizer state   : ✅")
print("Scheduler state   : ✅")
print("AMP state         : ✅")
print("History           : ✅")
print("Checkpoint dir    : ✅")
print("Dataset counts    : ✅")
print("Experiment state  : ✅")


# ============================================================
# STEP 22 COMPLETE
# ============================================================

print()
print("============================================================")
print("✅ STEP 22 COMPLETE")
print("============================================================")
print("Fresh model          : ✅")
print("Fresh loss           : ✅")
print("Fresh optimizer      : ✅")
print("LR scheduler         : ✅")
print("Fresh AMP scaler     : ✅")
print("Training history     : ✅")
print("Checkpoint tracking  : ✅")
print("Experiment state     : ✅")
print("No training performed: ✅")
print("============================================================")
print()
print("Ready for STEP 23 — REAL TRAINING")
print("============================================================")

## Updated training workflow — 50 epochs in 10 parts

Run the setup cells through **STEP 22** first. Then run the training helper and Parts 1–10 in order. Each part covers five epochs, writes an epoch checkpoint, updates the CSV history, and resumes from the latest checkpoint if the runtime reconnects. Files under `/kaggle/working` are runtime storage; download or save checkpoints to persistent storage before the Kaggle session is lost.

**Important:** this edit updates the training section only. The data-loading cells above still target BraTS 2020; they must be migrated and audited separately before using BraTS-SSA 2025.

In [ ]:
# ============================================================
# STEP 23 — RESUMABLE TRAINING HELPER (RUN THIS CELL FIRST)
# Training is split into 10 parts of 5 epochs each.
# Uses the existing train_one_epoch_21b / validate_one_epoch_21b.
# ============================================================

from pathlib import Path
import csv
import torch

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_10parts")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_CSV = Path("/kaggle/working/training_history.csv")
TOTAL_EPOCHS = 50

def _checkpoint_path(epoch):
    return CHECKPOINT_DIR / f"checkpoint_epoch_{epoch:03d}.pth"

def _load_training_checkpoint(path):
    state = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state["model_state_dict"])
    optimizer.load_state_dict(state["optimizer_state_dict"])
    if scheduler is not None and state.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(state["scheduler_state_dict"])
    if scaler is not None and state.get("scaler_state_dict") is not None:
        scaler.load_state_dict(state["scaler_state_dict"])
    history.update(state.get("history", history))
    return int(state["epoch"]), float(state.get("best_mean_dice", -1.0)), int(state.get("best_epoch", 0))

def run_epoch_part(part_number):
    if not 1 <= part_number <= 10:
        raise ValueError("part_number must be from 1 to 10")

    part_start = (part_number - 1) * 5 + 1
    part_end = part_number * 5
    latest_epoch = 0
    best_mean_dice = -1.0
    best_epoch = 0

    # Automatically resume from the newest available epoch checkpoint.
    existing = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pth"))
    if existing:
        epochs = []
        for p in existing:
            try:
                epochs.append((int(p.stem.split("_")[-1]), p))
            except ValueError:
                pass
        if epochs:
            latest_epoch, checkpoint_file = max(epochs, key=lambda item: item[0])
            latest_epoch, best_mean_dice, best_epoch = _load_training_checkpoint(checkpoint_file)
            print(f"Resumed from epoch {latest_epoch}: {checkpoint_file}")

    # Do not retrain completed epochs when a cell is re-run after reconnect.
    start_epoch = max(part_start, latest_epoch + 1)
    if start_epoch > part_end:
        print(f"Part {part_number}/10 already complete (epochs {part_start}-{part_end}). Skipping.")
        return

    for epoch in range(start_epoch, part_end + 1):
        train_result = train_one_epoch_21b(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            max_batches=None,
        )
        val_result = validate_one_epoch_21b(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            max_batches=None,
        )

        train_loss = float(train_result["loss"])
        val_loss = float(val_result["loss"])
        wt_dice = float(val_result["dice_wt"])
        tc_dice = float(val_result["dice_tc"])
        et_dice = float(val_result["dice_et"])
        mean_dice = float(val_result["dice_mean"])

        # ReduceLROnPlateau is configured with mode='max'.
        if scheduler is not None:
            scheduler.step(mean_dice)

        if mean_dice > best_mean_dice:
            best_mean_dice = mean_dice
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            torch.save(
                {
                    "epoch": epoch,
                    "best_mean_dice": best_mean_dice,
                    "best_epoch": best_epoch,
                    "model_state_dict": best_state,
                },
                CHECKPOINT_DIR / "best_model.pth",
            )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["wt_dice"].append(wt_dice)
        history["tc_dice"].append(tc_dice)
        history["et_dice"].append(et_dice)
        history["mean_dice"].append(mean_dice)
        history["learning_rate"].append(float(optimizer.param_groups[0]["lr"]))
        history["epoch_time"].append(float(train_result.get("time", 0.0) + val_result.get("time", 0.0)))

        state = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
            "best_mean_dice": best_mean_dice,
            "best_epoch": best_epoch,
        }
        temp_path = CHECKPOINT_DIR / f"checkpoint_epoch_{epoch:03d}.tmp"
        torch.save(state, temp_path)
        temp_path.replace(_checkpoint_path(epoch))

        # Keep a convenient latest checkpoint too.
        latest_tmp = CHECKPOINT_DIR / "last_model.tmp"
        torch.save(state, latest_tmp)
        latest_tmp.replace(CHECKPOINT_DIR / "last_model.pth")

        with HISTORY_CSV.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_loss", "val_loss", "wt_dice", "tc_dice", "et_dice", "mean_dice", "learning_rate", "epoch_time"])
            for i, e in enumerate(history["epoch"]):
                writer.writerow([
                    e, history["train_loss"][i], history["val_loss"][i],
                    history["wt_dice"][i], history["tc_dice"][i],
                    history["et_dice"][i], history["mean_dice"][i],
                    history["learning_rate"][i], history["epoch_time"][i]
                ])

        print(
            f"Epoch {epoch:02d}/{TOTAL_EPOCHS} | "
            f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
            f"WT={wt_dice:.4f} TC={tc_dice:.4f} ET={et_dice:.4f} "
            f"mean_Dice={mean_dice:.4f} | best={best_mean_dice:.4f} (epoch {best_epoch})"
        )

    print(f"Part {part_number}/10 finished: epochs {part_start}-{part_end}.")
    print("Latest checkpoint:", _checkpoint_path(part_end) if _checkpoint_path(part_end).exists() else "latest available checkpoint")
    print("History CSV:", HISTORY_CSV)

print("Training helper ready. Run Part 1, then Parts 2–10 in order.")


In [ ]:
# ============================================================
# STEP 23.1 — TRAINING PART 1/10: EPOCHS 1–5
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(1)

In [ ]:
# ============================================================
# STEP 23.2 — TRAINING PART 2/10: EPOCHS 6–10
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(2)

In [ ]:
# ============================================================
# STEP 23.3 — TRAINING PART 3/10: EPOCHS 11–15
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(3)

In [ ]:
# ============================================================
# STEP 23.4 — TRAINING PART 4/10: EPOCHS 16–20
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(4)

In [ ]:
# ============================================================
# STEP 23.5 — TRAINING PART 5/10: EPOCHS 21–25
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(5)

In [ ]:
# ============================================================
# STEP 23.6 — TRAINING PART 6/10: EPOCHS 26–30
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(6)

In [ ]:
# ============================================================
# STEP 23.7 — TRAINING PART 7/10: EPOCHS 31–35
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(7)

In [ ]:
# ============================================================
# STEP 23.8 — TRAINING PART 8/10: EPOCHS 36–40
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(8)

In [ ]:
# ============================================================
# STEP 23.9 — TRAINING PART 9/10: EPOCHS 41–45
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(9)

In [ ]:
# ============================================================
# STEP 23.10 — TRAINING PART 10/10: EPOCHS 46–50
# Run this cell after the training helper and preceding parts.
# If a checkpoint already exists, completed epochs are automatically skipped.
# ============================================================

run_epoch_part(10)

In [ ]:

# ============================================================
# STEP 24 — COMPREHENSIVE EVALUATION METRICS
# 3D RCNN-U-Net V2 | BraTS 2020 / BraTS-GLI 2023
#
# Metrics: Dice, HD95, IoU, Precision, Recall, Specificity
# Regions: WT, TC, ET
# Includes: CSV logging + synthetic smoke test
# ============================================================

import csv
from pathlib import Path

import numpy as np
import torch
from scipy.ndimage import distance_transform_edt, binary_erosion


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

METRICS_CSV = Path(
    "/kaggle/working/step24_patient_metrics.csv"
)

# Set this to the actual voxel spacing from your MRI header
# before reporting physical HD95 values.
DEFAULT_SPACING = (1.0, 1.0, 1.0)


# ------------------------------------------------------------
# 2. CONVERT MASK TO NUMPY BINARY MASK
# ------------------------------------------------------------

def to_binary_mask(mask, threshold=0.5):
    if torch.is_tensor(mask):
        mask = mask.detach().float().cpu().numpy()

    mask = np.asarray(mask)

    if not np.isfinite(mask).all():
        raise ValueError("Mask contains NaN or infinite values.")

    return mask >= threshold


# ------------------------------------------------------------
# 3. HD95 — 95TH PERCENTILE HAUSDORFF DISTANCE
# ------------------------------------------------------------

def calculate_hd95(pred, target, spacing=DEFAULT_SPACING):
    pred = np.asarray(pred, dtype=bool)
    target = np.asarray(target, dtype=bool)

    if not pred.any() and not target.any():
        return 0.0

    if not pred.any() or not target.any():
        return float("nan")

    pred_surface = pred ^ binary_erosion(
        pred, iterations=1, border_value=0
    )
    target_surface = target ^ binary_erosion(
        target, iterations=1, border_value=0
    )

    dist_to_target = distance_transform_edt(
        ~target_surface, sampling=spacing
    )
    dist_to_pred = distance_transform_edt(
        ~pred_surface, sampling=spacing
    )

    distances = np.concatenate([
        dist_to_target[pred_surface],
        dist_to_pred[target_surface],
    ])

    return float(np.percentile(distances, 95))


# ------------------------------------------------------------
# 4. METRICS FOR ONE 3D REGION
# ------------------------------------------------------------

def calculate_binary_metrics(
    prediction,
    ground_truth,
    spacing=DEFAULT_SPACING,
):
    pred = np.squeeze(to_binary_mask(prediction))
    target = np.squeeze(to_binary_mask(ground_truth))

    if pred.shape != target.shape:
        raise ValueError(
            f"Shape mismatch: prediction={pred.shape}, "
            f"ground_truth={target.shape}"
        )

    if pred.ndim != 3:
        raise ValueError(
            f"Expected a 3D mask after squeeze; got {pred.shape}"
        )

    if len(spacing) != 3 or any(s <= 0 for s in spacing):
        raise ValueError("Spacing must contain 3 positive values.")

    tp = int(np.logical_and(pred, target).sum())
    fp = int(np.logical_and(pred, ~target).sum())
    fn = int(np.logical_and(~pred, target).sum())
    tn = int(np.logical_and(~pred, ~target).sum())

    pred_count = int(pred.sum())
    target_count = int(target.sum())

    if pred_count == 0 and target_count == 0:
        dice = 1.0
        iou = 1.0
        precision = 1.0
        recall = 1.0
        hd95 = 0.0
    else:
        dice = (
            2 * tp / (2 * tp + fp + fn)
            if (2 * tp + fp + fn) else 0.0
        )

        iou = (
            tp / (tp + fp + fn)
            if (tp + fp + fn) else 0.0
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) else 0.0
        )

        hd95 = calculate_hd95(pred, target, spacing)

    specificity = (
        tn / (tn + fp)
        if (tn + fp) else float("nan")
    )

    return {
        "dice": float(dice),
        "hd95": float(hd95),
        "iou": float(iou),
        "precision": float(precision),
        "recall_sensitivity": float(recall),
        "specificity": float(specificity),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


# ------------------------------------------------------------
# 5. METRICS FOR WT, TC, ET
# ------------------------------------------------------------

def calculate_wt_tc_et_metrics(
    predictions,
    targets,
    spacing=DEFAULT_SPACING,
):
    """
    predictions and targets must be dictionaries containing
    WT, TC and ET. Each value must be a corresponding 3D mask
    or probability map.
    """

    regions = ("WT", "TC", "ET")
    results = {}

    for region in regions:
        if region not in predictions or region not in targets:
            raise KeyError(
                f"Missing {region}. Provide WT, TC and ET masks."
            )

        results[region] = calculate_binary_metrics(
            predictions[region],
            targets[region],
            spacing=spacing,
        )

    metric_names = [
        "dice",
        "hd95",
        "iou",
        "precision",
        "recall_sensitivity",
        "specificity",
    ]

    results["MEAN"] = {}

    for metric in metric_names:
        values = np.array(
            [results[r][metric] for r in regions],
            dtype=float,
        )

        results["MEAN"][metric] = (
            float(np.nanmean(values))
            if np.isfinite(values).any()
            else float("nan")
        )

    return results


# ------------------------------------------------------------
# 6. SAVE PER-PATIENT METRICS TO CSV
# ------------------------------------------------------------

METRIC_FIELDS = [
    "patient_id",
    "region",
    "dice",
    "hd95",
    "iou",
    "precision",
    "recall_sensitivity",
    "specificity",
    "tp",
    "fp",
    "fn",
    "tn",
]


def save_patient_metrics(
    patient_id,
    results,
    csv_path=METRICS_CSV,
):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    write_header = not csv_path.exists()

    with csv_path.open("a", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=METRIC_FIELDS
        )

        if write_header:
            writer.writeheader()

        for region in ("WT", "TC", "ET"):
            row = {
                "patient_id": patient_id,
                "region": region,
            }
            row.update(results[region])
            writer.writerow(row)


# ------------------------------------------------------------
# 7. SMOKE TEST — SYNTHETIC DATA, NO GPU REQUIRED
# ------------------------------------------------------------

shape = (16, 16, 16)

test_mask = np.zeros(shape, dtype=np.uint8)
test_mask[4:10, 4:10, 4:10] = 1

test_predictions = {
    "WT": test_mask.copy(),
    "TC": test_mask.copy(),
    "ET": test_mask.copy(),
}

test_targets = {
    "WT": test_mask.copy(),
    "TC": test_mask.copy(),
    "ET": test_mask.copy(),
}

test_results = calculate_wt_tc_et_metrics(
    test_predictions,
    test_targets,
    spacing=DEFAULT_SPACING,
)

for region in ("WT", "TC", "ET"):
    assert abs(test_results[region]["dice"] - 1.0) < 1e-8
    assert abs(test_results[region]["iou"] - 1.0) < 1e-8
    assert abs(test_results[region]["hd95"]) < 1e-8

# Test CSV output
save_patient_metrics(
    "synthetic_smoke_test",
    test_results,
)

print("=" * 60)
print("STEP 24 METRICS SMOKE TEST PASSED")
print("=" * 60)

for region in ("WT", "TC", "ET"):
    print(
        f"{region}: "
        f"Dice={test_results[region]['dice']:.4f}, "
        f"HD95={test_results[region]['hd95']:.4f}, "
        f"IoU={test_results[region]['iou']:.4f}, "
        f"Precision={test_results[region]['precision']:.4f}, "
        f"Recall={test_results[region]['recall_sensitivity']:.4f}, "
        f"Specificity={test_results[region]['specificity']:.4f}"
    )

print("\nCSV path:", METRICS_CSV)
print("CSV exists:", METRICS_CSV.exists())
print(
    "\nNOTE: This was a synthetic test, not a real model evaluation."
)


In [ ]:

# ============================================================
# STEP 25: PATIENT-LEVEL VALIDATION METRICS INTEGRATION
# 3D RCNN-U-Net V2 | BraTS 2020 / BraTS-GLI 2023
# Requires the metric functions from Step 24
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

STEP25_CSV = Path("/kaggle/working/step25_validation_metrics.csv")

REGIONS = ("WT", "TC", "ET")


def validate_patient_step25(
    patient_id,
    predictions,
    targets,
    spacing=(1.0, 1.0, 1.0),
    save_csv=True
):
    """
    Evaluate one patient.

    predictions: dict with WT, TC, ET 3D binary masks
    targets:     dict with WT, TC, ET 3D ground-truth masks
    spacing:     voxel spacing in the same order as array axes

    IMPORTANT:
    Convert model outputs to the correct WT/TC/ET masks
    before calling this function.
    """

    if not isinstance(predictions, dict) or not isinstance(targets, dict):
        raise TypeError(
            "predictions and targets must be dictionaries "
            "with WT, TC and ET keys."
        )

    for region in REGIONS:
        if region not in predictions or region not in targets:
            raise KeyError(
                f"Missing region '{region}'. "
                "Both dictionaries must contain WT, TC and ET."
            )

    # Calculate Step 24 metrics
    results = calculate_wt_tc_et_metrics(
        predictions=predictions,
        targets=targets,
        spacing=spacing
    )

    rows = []

    for region in REGIONS:
        metrics = results[region]

        row = {
            "patient_id": str(patient_id),
            "region": region,
            "dice": metrics.get("dice", np.nan),
            "hd95": metrics.get("hd95", np.nan),
            "iou": metrics.get("iou", np.nan),
            "precision": metrics.get("precision", np.nan),
            "recall_sensitivity": metrics.get(
                "recall_sensitivity",
                metrics.get("recall", np.nan)
            ),
            "specificity": metrics.get("specificity", np.nan),
        }

        rows.append(row)

    # Calculate mean across WT, TC, ET for each metric.
    # NaN values are ignored; this is an arithmetic region mean.
    metric_names = [
        "dice", "hd95", "iou", "precision",
        "recall_sensitivity", "specificity"
    ]

    mean_row = {"patient_id": str(patient_id), "region": "MEAN"}

    for metric_name in metric_names:
        values = [
            row[metric_name]
            for row in rows
            if np.isfinite(row[metric_name])
        ]
        mean_row[metric_name] = (
            float(np.mean(values)) if values else np.nan
        )

    rows.append(mean_row)
    patient_df = pd.DataFrame(rows)

    if save_csv:
        STEP25_CSV.parent.mkdir(parents=True, exist_ok=True)

        write_header = not STEP25_CSV.exists()

        patient_df.to_csv(
            STEP25_CSV,
            mode="a",
            header=write_header,
            index=False
        )

    print(f"\nPatient: {patient_id}")
    print(patient_df.to_string(index=False))

    if save_csv:
        print(f"\nMetrics saved to: {STEP25_CSV}")

    return results, patient_df


# ------------------------------------------------------------
# STEP 25 SMOKE TEST
# Synthetic masks only — NOT actual model evaluation
# ------------------------------------------------------------

test_mask = np.zeros((16, 16, 16), dtype=np.uint8)
test_mask[4:10, 4:10, 4:10] = 1

test_predictions = {
    "WT": test_mask.copy(),
    "TC": test_mask.copy(),
    "ET": test_mask.copy()
}

test_targets = {
    "WT": test_mask.copy(),
    "TC": test_mask.copy(),
    "ET": test_mask.copy()
}

test_results, test_df = validate_patient_step25(
    patient_id="STEP25_SYNTHETIC_TEST",
    predictions=test_predictions,
    targets=test_targets,
    spacing=(1.0, 1.0, 1.0),
    save_csv=False
)

for region in REGIONS:
    assert abs(test_results[region]["dice"] - 1.0) < 1e-6
    assert abs(test_results[region]["iou"] - 1.0) < 1e-6
    assert abs(test_results[region]["hd95"]) < 1e-6

print("\nSTEP 25 SMOKE TEST PASSED")
print("Perfect synthetic masks produced Dice=1, IoU=1 and HD95=0.")
print("No actual model or patient data was evaluated in this test.")


In [ ]:

# ============================================================
# STEP 26: INSPECT EXISTING MODEL AND VALIDATION PIPELINE
# ============================================================

import inspect
import torch

print("=" * 65)
print("STEP 26: VALIDATION PIPELINE INSPECTION")
print("=" * 65)

# 1. Check common model variable names
model_candidates = [
    "model", "net", "unet", "rcnn_unet", "model_v2"
]

found_models = []

for name in model_candidates:
    obj = globals().get(name)

    if isinstance(obj, torch.nn.Module):
        found_models.append((name, obj))
        print(f"\nModel found: {name}")
        print(f"Model class: {obj.__class__.__name__}")
        print(f"Device: {next(obj.parameters(), torch.empty(0)).device}")

if not found_models:
    print("\nNo model found under common variable names.")
    print("We will need the actual model variable name.")

# 2. Look for existing validation functions
validation_candidates = [
    "validate", "validation", "validate_model",
    "validate_one_epoch", "run_validation",
    "validation_step", "evaluate"
]

found_functions = []

for name in validation_candidates:
    obj = globals().get(name)

    if callable(obj):
        try:
            signature = inspect.signature(obj)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        found_functions.append((name, obj))
        print(f"\nValidation candidate: {name}")
        print(f"Signature: {signature}")

        try:
            source = inspect.getsource(obj)
            print("\nFunction source (first 60 lines):")
            print("\n".join(source.splitlines()[:60]))
        except (OSError, TypeError):
            print("Source unavailable; inspect its notebook cell manually.")

# 3. Check metric helpers from Steps 24 and 25
required_helpers = [
    "calculate_binary_metrics",
    "calculate_hd95",
    "calculate_wt_tc_et_metrics",
    "validate_patient_step25"
]

print("\n" + "=" * 65)
print("METRIC HELPER CHECK")
print("=" * 65)

for name in required_helpers:
    print(f"{name}: {'AVAILABLE' if callable(globals().get(name)) else 'NOT FOUND'}")

# 4. Reminder about the next integration step
print("\n" + "=" * 65)
print("INSPECTION COMPLETE")
print("=" * 65)
print("Do not evaluate real patients until we verify:")
print("- model output shape and activation")
print("- BraTS ground-truth label mapping")
print("- validation batch structure")
print("- actual voxel spacing")
print("\nShare this output so we can integrate the metrics safely.")


In [ ]:

# ============================================================
# STEP 27: REAL VALIDATION BATCH SMOKE TEST
# 3D RCNN-U-Net V2 | No training / no weight updates
# ============================================================

import torch
import numpy as np

print("=" * 65)
print("STEP 27: REAL VALIDATION BATCH SMOKE TEST")
print("=" * 65)

# ------------------------------------------------------------
# 1. Find an existing model
# ------------------------------------------------------------
model_names = ["model", "net", "unet", "rcnn_unet", "model_v2"]

model = next(
    (
        globals()[name]
        for name in model_names
        if name in globals()
        and isinstance(globals()[name], torch.nn.Module)
    ),
    None
)

if model is None:
    raise RuntimeError(
        "Model not found. Check the model variable name in your notebook."
    )

# ------------------------------------------------------------
# 2. Find an existing validation DataLoader
# ------------------------------------------------------------
loader_names = [
    "val_loader", "valid_loader", "validation_loader",
    "val_dataloader", "validation_dataloader"
]

val_loader = next(
    (
        globals()[name]
        for name in loader_names
        if name in globals()
        and hasattr(globals()[name], "__iter__")
    ),
    None
)

if val_loader is None:
    raise RuntimeError(
        "Validation DataLoader not found. "
        "Check whether its variable is named differently."
    )

# ------------------------------------------------------------
# 3. Fetch one batch
# ------------------------------------------------------------
batch = next(iter(val_loader))

print("\nBatch type:", type(batch).__name__)

if isinstance(batch, dict):
    print("Batch keys:", list(batch.keys()))
    for key, value in batch.items():
        if torch.is_tensor(value):
            print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
        else:
            print(f"{key}: type={type(value).__name__}")

elif isinstance(batch, (tuple, list)):
    print("Number of batch elements:", len(batch))
    for index, value in enumerate(batch):
        if torch.is_tensor(value):
            print(
                f"batch[{index}]: shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(f"batch[{index}]: type={type(value).__name__}")

else:
    raise TypeError(
        "Unsupported batch format. Inspect the dataset's __getitem__ method."
    )

# ------------------------------------------------------------
# 4. Extract input tensor conservatively
# ------------------------------------------------------------
if isinstance(batch, dict):
    input_candidates = [
        "image", "images", "input", "inputs", "volume", "data", "x"
    ]
    input_tensor = next(
        (
            batch[key] for key in input_candidates
            if key in batch and torch.is_tensor(batch[key])
        ),
        None
    )
elif isinstance(batch, (tuple, list)) and len(batch) >= 2:
    input_tensor = batch[0] if torch.is_tensor(batch[0]) else None
else:
    input_tensor = None

if input_tensor is None:
    raise RuntimeError(
        "Could not identify the input image tensor safely. "
        "Use the printed batch structure to map the correct key."
    )

if input_tensor.ndim < 4:
    raise ValueError(
        f"Unexpected input dimensions: {tuple(input_tensor.shape)}"
    )

# ------------------------------------------------------------
# 5. Run one forward pass without gradient tracking
# ------------------------------------------------------------
try:
    device = next(model.parameters()).device
except StopIteration:
    device = input_tensor.device

input_tensor = input_tensor.to(device)

was_training = model.training
model.eval()

try:
    with torch.no_grad():
        output = model(input_tensor)
finally:
    model.train(was_training)

# ------------------------------------------------------------
# 6. Inspect output without assuming its encoding
# ------------------------------------------------------------
print("\nModel class:", model.__class__.__name__)
print("Input shape:", tuple(input_tensor.shape))
print("Output type:", type(output).__name__)

def report_output(obj, prefix="output"):
    if torch.is_tensor(obj):
        print(
            f"{prefix}: shape={tuple(obj.shape)}, "
            f"dtype={obj.dtype}, device={obj.device}"
        )
        if obj.numel() > 0 and torch.is_floating_point(obj):
            finite = torch.isfinite(obj)
            print("  All values finite:", bool(finite.all().item()))
            if bool(finite.any().item()):
                finite_values = obj[finite]
                print(
                    "  Finite value range:",
                    float(finite_values.min().item()),
                    "to",
                    float(finite_values.max().item())
                )
    elif isinstance(obj, dict):
        print(f"{prefix} keys:", list(obj.keys()))
        for key, value in obj.items():
            report_output(value, f"{prefix}[{key!r}]")
    elif isinstance(obj, (tuple, list)):
        print(f"{prefix} contains {len(obj)} elements")
        for index, value in enumerate(obj):
            report_output(value, f"{prefix}[{index}]")
    else:
        print(f"{prefix}: unsupported output object type")

report_output(output)

print("\n" + "=" * 65)
print("STEP 27 FORWARD-PASS TEST COMPLETED")
print("No optimizer step or parameter update was performed.")
print("Ground-truth mapping and region metrics are not evaluated yet.")
print("=" * 65)


In [ ]:

# ============================================================
# STEP 28: BRATS GROUND-TRUTH LABEL MAPPING & VERIFICATION
# BraTS 2020 raw labels: 0, 1, 2, 4
# ============================================================

import numpy as np
import torch

def brats_label_to_regions(segmentation):
    """
    Convert a RAW BraTS segmentation label map into WT, TC, ET masks.

    Raw BraTS label convention:
        0 = background
        1 = NCR/NET
        2 = edema
        4 = enhancing tumor

    Input:
        3D NumPy array or Torch tensor containing raw labels.

    Output:
        Dictionary of 3D uint8 binary masks: WT, TC, ET.
    """

    if torch.is_tensor(segmentation):
        segmentation = segmentation.detach().cpu().numpy()

    segmentation = np.asarray(segmentation)

    # Remove only singleton dimensions; preserve the 3D volume.
    segmentation = np.squeeze(segmentation)

    if segmentation.ndim != 3:
        raise ValueError(
            f"Expected a 3D raw segmentation map; "
            f"received shape {segmentation.shape}"
        )

    if not np.isfinite(segmentation).all():
        raise ValueError("Segmentation contains NaN or infinite values.")

    labels = set(np.unique(segmentation).tolist())
    allowed_labels = {0, 1, 2, 4}

    if not labels.issubset(allowed_labels):
        raise ValueError(
            f"Unexpected labels: {sorted(labels - allowed_labels)}. "
            "This function expects raw BraTS labels 0, 1, 2, 4. "
            "Check your dataset's label encoding before proceeding."
        )

    wt = np.isin(segmentation, [1, 2, 4])
    tc = np.isin(segmentation, [1, 4])
    et = segmentation == 4

    regions = {
        "WT": wt.astype(np.uint8),
        "TC": tc.astype(np.uint8),
        "ET": et.astype(np.uint8),
    }

    # Validate nesting: ET must be inside TC, and TC inside WT.
    assert np.all(regions["ET"] <= regions["TC"])
    assert np.all(regions["TC"] <= regions["WT"])

    return regions


def verify_brats_regions(regions, name="segmentation"):
    """Print region sizes and verify WT/TC/ET nesting."""

    for key in ("WT", "TC", "ET"):
        if key not in regions:
            raise KeyError(f"{name}: missing region {key}")

        mask = np.asarray(regions[key])

        if mask.ndim != 3:
            raise ValueError(
                f"{name}/{key} must be 3D; got {mask.shape}"
            )

        if not np.isin(mask, [0, 1]).all():
            raise ValueError(
                f"{name}/{key} must be a binary mask containing 0 and 1."
            )

    wt, tc, et = (regions[k].astype(bool) for k in ("WT", "TC", "ET"))

    if np.any(et & ~tc):
        raise ValueError("ET contains voxels outside TC.")

    if np.any(tc & ~wt):
        raise ValueError("TC contains voxels outside WT.")

    print(f"\n{name}: region verification PASSED")
    for key in ("WT", "TC", "ET"):
        mask = np.asarray(regions[key])
        print(
            f"{key}: shape={mask.shape}, "
            f"positive_voxels={int(mask.sum())}"
        )


# ------------------------------------------------------------
# Synthetic test — no patient data is modified
# ------------------------------------------------------------

synthetic_seg = np.zeros((12, 12, 12), dtype=np.uint8)
synthetic_seg[2:8, 2:8, 2:8] = 2
synthetic_seg[3:6, 3:6, 3:6] = 1
synthetic_seg[4:5, 4:5, 4:5] = 4

synthetic_regions = brats_label_to_regions(synthetic_seg)
verify_brats_regions(synthetic_regions, "Synthetic BraTS test")

assert synthetic_regions["WT"].sum() == 216
assert synthetic_regions["TC"].sum() == 28
assert synthetic_regions["ET"].sum() == 1

print("\nSTEP 28 SMOKE TEST PASSED")
print("Raw labels were converted to nested WT/TC/ET masks correctly.")
print("Synthetic data only; no actual patient was evaluated.")


In [ ]:

# ============================================================
# STEP 29: MODEL PREDICTION MAPPING & REGION VERIFICATION
# ============================================================

import numpy as np
import torch

REGION_NAMES = ("WT", "TC", "ET")


def prediction_to_regions(
    prediction,
    output_format,
    channel_order=("WT", "TC", "ET"),
    thresholds=(0.5, 0.5, 0.5),
):
    """
    Convert one patient's model output into 3D WT/TC/ET masks.

    Supported output_format values:
      "binary_channels":
          Three channels already contain binary 0/1 masks.
      "probability_channels":
          Three channels contain WT/TC/ET probabilities.
      "logit_channels":
          Three channels contain WT/TC/ET logits.
      "multiclass_labels":
          One 3D volume contains BraTS labels 0, 1, 2, 4.

    Expected channel layout for channel-based formats:
      (3, D, H, W)

    This function does not handle batched tensors or deep-supervision
    outputs automatically. Select one patient's output first.
    """

    if torch.is_tensor(prediction):
        prediction = prediction.detach().float().cpu().numpy()

    prediction = np.asarray(prediction)
    prediction = np.squeeze(prediction)

    if not np.isfinite(prediction).all():
        raise ValueError("Prediction contains NaN or infinite values.")

    if output_format == "multiclass_labels":
        return brats_label_to_regions(prediction)

    if output_format not in {
        "binary_channels",
        "probability_channels",
        "logit_channels",
    }:
        raise ValueError(
            "Unsupported output_format. Choose one of: "
            "binary_channels, probability_channels, "
            "logit_channels, multiclass_labels."
        )

    if prediction.ndim != 4:
        raise ValueError(
            "Channel-based output must have shape (C,D,H,W) "
            f"for one patient. Received {prediction.shape}. "
            "If your output is (B,C,D,H,W), select one batch item first."
        )

    if prediction.shape[0] != 3:
        raise ValueError(
            f"Expected 3 output channels, got {prediction.shape[0]}. "
            "Inspect the model architecture and output head."
        )

    if len(channel_order) != 3 or set(channel_order) != set(REGION_NAMES):
        raise ValueError("channel_order must contain WT, TC and ET exactly once.")

    if len(thresholds) != 3:
        raise ValueError("Provide one threshold per channel.")

    channel_masks = {}

    for index, region in enumerate(channel_order):
        values = prediction[index]

        if output_format == "binary_channels":
            if not np.isin(values, [0, 1]).all():
                raise ValueError(
                    f"{region} channel is not binary. "
                    "Choose probability_channels or logit_channels if appropriate."
                )
            mask = values.astype(bool)

        elif output_format == "probability_channels":
            if values.min() < 0 or values.max() > 1:
                raise ValueError(
                    f"{region} values are outside [0,1]. "
                    "Verify whether sigmoid was applied."
                )
            mask = values >= float(thresholds[index])

        else:  # logits
            # sigmoid(logit) >= threshold is equivalent to:
            # logit >= log(threshold / (1 - threshold))
            threshold = float(thresholds[index])
            if not 0 < threshold < 1:
                raise ValueError("Logit thresholds must be between 0 and 1.")
            logit_threshold = np.log(threshold / (1 - threshold))
            mask = values >= logit_threshold

        channel_masks[region] = mask.astype(np.uint8)

    # Do not silently change predictions to force nested regions.
    # Instead, flag whether the model output satisfies WT/TC/ET nesting.
    wt = channel_masks["WT"].astype(bool)
    tc = channel_masks["TC"].astype(bool)
    et = channel_masks["ET"].astype(bool)

    nesting_valid = (
        not np.any(tc & ~wt)
        and not np.any(et & ~tc)
    )

    print("Prediction shape:", prediction.shape)
    print("Output format:", output_format)
    print("WT/TC/ET nesting valid:", nesting_valid)

    if not nesting_valid:
        print(
            "WARNING: predicted masks are not nested. "
            "Check target encoding, channel order and model output."
        )

    return channel_masks


def inspect_model_output(output):
    """Report tensor shapes and value ranges without changing the output."""

    if torch.is_tensor(output):
        tensor = output.detach()
        print("Tensor shape:", tuple(tensor.shape))
        print("Tensor dtype:", tensor.dtype)

        if tensor.numel():
            print("All finite:", bool(torch.isfinite(tensor).all().item()))

            if torch.is_floating_point(tensor):
                finite = tensor[torch.isfinite(tensor)]
                if finite.numel():
                    print(
                        "Finite value range:",
                        float(finite.min().item()),
                        "to",
                        float(finite.max().item())
                    )

    elif isinstance(output, dict):
        print("Output dictionary keys:", list(output.keys()))
        for key, value in output.items():
            print(f"\nOutput[{key!r}]:")
            inspect_model_output(value)

    elif isinstance(output, (list, tuple)):
        print("Output sequence length:", len(output))
        for index, value in enumerate(output):
            print(f"\nOutput[{index}]:")
            inspect_model_output(value)

    else:
        print("Output type:", type(output).__name__)


# ------------------------------------------------------------
# Synthetic test: three already-binary channels
# ------------------------------------------------------------

test_prediction = np.zeros((3, 12, 12, 12), dtype=np.uint8)

# WT region
test_prediction[0, 2:8, 2:8, 2:8] = 1

# TC region, nested inside WT
test_prediction[1, 3:6, 3:6, 3:6] = 1

# ET region, nested inside TC
test_prediction[2, 4:5, 4:5, 4:5] = 1

test_regions = prediction_to_regions(
    test_prediction,
    output_format="binary_channels",
    channel_order=("WT", "TC", "ET"),
)

verify_brats_regions(test_regions, "Step 29 synthetic prediction")

assert test_regions["WT"].sum() == 216
assert test_regions["TC"].sum() == 27
assert test_regions["ET"].sum() == 1

print("\nSTEP 29 SMOKE TEST PASSED")
print("Synthetic predictions converted and verified successfully.")
print("No real model prediction or patient was evaluated.")


In [ ]:

# ============================================================
# STEP 30: PREDICTION VS GROUND-TRUTH EVALUATION
# Metrics: Dice, HD95, IoU, Precision, Recall, Specificity
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

STEP30_CSV = Path("/kaggle/working/step30_evaluation_metrics.csv")
REGIONS = ("WT", "TC", "ET")


def evaluate_prediction_step30(
    patient_id,
    predicted_regions,
    ground_truth_regions,
    spacing,
    save_csv=True,
):
    """
    Evaluate one patient's WT/TC/ET predictions.

    Both dictionaries must contain 3D binary masks.
    spacing must match the volume's axis order and use consistent units
    (normally millimeters for MRI voxel spacing).
    """

    # 1. Validate required regions
    for name, masks in (
        ("predictions", predicted_regions),
        ("ground truth", ground_truth_regions),
    ):
        if not isinstance(masks, dict):
            raise TypeError(f"{name} must be a dictionary.")

        for region in REGIONS:
            if region not in masks:
                raise KeyError(f"{name} is missing region {region}.")

            mask = np.asarray(masks[region])

            if mask.ndim != 3:
                raise ValueError(
                    f"{name}/{region} must be 3D; got {mask.shape}"
                )

            if not np.isin(mask, [0, 1]).all():
                raise ValueError(
                    f"{name}/{region} must contain binary values 0 and 1."
                )

    # 2. Calculate metrics using the Step 24 function
    results = calculate_wt_tc_et_metrics(
        predictions=predicted_regions,
        targets=ground_truth_regions,
        spacing=spacing,
    )

    # 3. Prepare patient-level rows
    rows = []

    for region in REGIONS:
        metric = results[region]

        rows.append({
            "patient_id": str(patient_id),
            "region": region,
            "dice": metric.get("dice", np.nan),
            "hd95": metric.get("hd95", np.nan),
            "iou": metric.get("iou", np.nan),
            "precision": metric.get("precision", np.nan),
            "recall_sensitivity": metric.get(
                "recall_sensitivity",
                metric.get("recall", np.nan),
            ),
            "specificity": metric.get("specificity", np.nan),
        })

    # 4. Arithmetic mean across regions.
    # HD95 is averaged only across finite values.
    mean_row = {"patient_id": str(patient_id), "region": "MEAN"}

    for key in (
        "dice", "hd95", "iou", "precision",
        "recall_sensitivity", "specificity",
    ):
        values = [
            row[key] for row in rows
            if np.isfinite(row[key])
        ]
        mean_row[key] = float(np.mean(values)) if values else np.nan

    rows.append(mean_row)
    df = pd.DataFrame(rows)

    # 5. Save CSV, avoiding duplicate headers
    if save_csv:
        STEP30_CSV.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(
            STEP30_CSV,
            mode="a",
            header=not STEP30_CSV.exists(),
            index=False,
        )

    print(f"\nEvaluation for patient: {patient_id}")
    print(df.to_string(index=False))

    if save_csv:
        print(f"\nSaved metrics to: {STEP30_CSV}")

    return results, df


# ------------------------------------------------------------
# SMOKE TEST: identical synthetic prediction and target
# ------------------------------------------------------------

synthetic_mask = np.zeros((16, 16, 16), dtype=np.uint8)
synthetic_mask[4:10, 4:10, 4:10] = 1

synthetic_prediction = {
    "WT": synthetic_mask.copy(),
    "TC": synthetic_mask.copy(),
    "ET": synthetic_mask.copy(),
}

synthetic_target = {
    "WT": synthetic_mask.copy(),
    "TC": synthetic_mask.copy(),
    "ET": synthetic_mask.copy(),
}

test_results, test_df = evaluate_prediction_step30(
    patient_id="STEP30_SYNTHETIC_TEST",
    predicted_regions=synthetic_prediction,
    ground_truth_regions=synthetic_target,
    spacing=(1.0, 1.0, 1.0),
    save_csv=False,
)

for region in REGIONS:
    assert abs(test_results[region]["dice"] - 1.0) < 1e-6
    assert abs(test_results[region]["iou"] - 1.0) < 1e-6
    assert abs(test_results[region]["hd95"]) < 1e-6

print("\nSTEP 30 SMOKE TEST PASSED")
print("Identical masks should produce Dice=1, IoU=1 and HD95=0.")
print("This is a synthetic test, not a model performance result.")


In [ ]:

# ============================================================
# STEP 31: FULL VALIDATION PIPELINE READINESS CHECK
# Checks existing model, validation loader and metric functions
# Does not train the model or alter model weights
# ============================================================

import torch
import inspect
from pathlib import Path

print("=" * 70)
print("STEP 31: FULL VALIDATION READINESS CHECK")
print("=" * 70)

# ------------------------------------------------------------
# 1. Find an existing model
# ------------------------------------------------------------
model_names = ["model", "net", "unet", "rcnn_unet", "model_v2"]

model_name = next(
    (
        name for name in model_names
        if name in globals()
        and isinstance(globals()[name], torch.nn.Module)
    ),
    None
)

if model_name is None:
    print("MODEL: NOT FOUND under common variable names.")
else:
    model = globals()[model_name]
    print(f"MODEL: {model_name}")
    print(f"Model class: {model.__class__.__name__}")
    print(f"Training mode currently: {model.training}")

# ------------------------------------------------------------
# 2. Find validation DataLoader
# ------------------------------------------------------------
loader_names = [
    "val_loader", "valid_loader", "validation_loader",
    "val_dataloader", "validation_dataloader"
]

loader_name = next(
    (
        name for name in loader_names
        if name in globals()
        and hasattr(globals()[name], "__iter__")
        and hasattr(globals()[name], "__len__")
    ),
    None
)

if loader_name is None:
    print("\nVALIDATION LOADER: NOT FOUND.")
else:
    val_loader = globals()[loader_name]
    print(f"\nVALIDATION LOADER: {loader_name}")
    print(f"Number of batches: {len(val_loader)}")

# ------------------------------------------------------------
# 3. Check required evaluation functions
# ------------------------------------------------------------
required_functions = [
    "calculate_binary_metrics",
    "calculate_hd95",
    "calculate_wt_tc_et_metrics",
    "brats_label_to_regions",
    "prediction_to_regions",
    "evaluate_prediction_step30",
]

print("\nMETRIC FUNCTION STATUS")
print("-" * 70)

functions_ready = True

for function_name in required_functions:
    available = callable(globals().get(function_name))
    print(f"{function_name}: {'AVAILABLE' if available else 'MISSING'}")
    if not available:
        functions_ready = False

# ------------------------------------------------------------
# 4. Check whether a validation function already exists
# ------------------------------------------------------------
validation_function_names = [
    "validate",
    "validate_model",
    "validate_one_epoch",
    "run_validation",
    "evaluate",
]

print("\nEXISTING VALIDATION FUNCTIONS")
print("-" * 70)

validation_functions_found = False

for function_name in validation_function_names:
    function = globals().get(function_name)

    if callable(function):
        validation_functions_found = True
        try:
            signature = inspect.signature(function)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"{function_name}{signature}")

if not validation_functions_found:
    print("No common validation function name found.")

# ------------------------------------------------------------
# 5. Readiness summary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 31 READINESS SUMMARY")
print("=" * 70)

print("Model available:", model_name is not None)
print("Validation loader available:", loader_name is not None)
print("All metric functions available:", functions_ready)

if (
    model_name is not None
    and loader_name is not None
    and functions_ready
):
    print(
        "\nREADY FOR THE NEXT INTEGRATION STAGE."
        "\nBefore full evaluation, verify the model output format,"
        "\nground-truth encoding and MRI voxel spacing."
    )
else:
    print(
        "\nNOT READY FOR FULL VALIDATION YET."
        "\nUse the missing items above to identify what needs fixing."
    )

print("\nNo training, optimizer step, or model-weight update was performed.")


In [ ]:

# ============================================================
# STEP 32: INSPECT REAL VALIDATION BATCH AND MODEL OUTPUT
# ============================================================

import torch
import numpy as np

print("=" * 70)
print("STEP 32: REAL VALIDATION BATCH INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate existing model and validation loader
# ------------------------------------------------------------

model_names = ["model", "net", "unet", "rcnn_unet", "model_v2"]
loader_names = [
    "val_loader", "valid_loader", "validation_loader",
    "val_dataloader", "validation_dataloader"
]

model_name = next(
    (
        n for n in model_names
        if n in globals()
        and isinstance(globals()[n], torch.nn.Module)
    ),
    None
)

loader_name = next(
    (
        n for n in loader_names
        if n in globals()
        and hasattr(globals()[n], "__iter__")
    ),
    None
)

if model_name is None:
    raise RuntimeError(
        "Could not find the model. Set model_name to your actual "
        "model variable name."
    )

if loader_name is None:
    raise RuntimeError(
        "Could not find the validation DataLoader. Set loader_name "
        "to your actual DataLoader variable name."
    )

model = globals()[model_name]
val_loader = globals()[loader_name]

print("Model variable:", model_name)
print("Validation loader:", loader_name)

# ------------------------------------------------------------
# 2. Fetch one batch and inspect its structure
# ------------------------------------------------------------

batch = next(iter(val_loader))

def inspect_value(name, value):
    if torch.is_tensor(value):
        print(
            f"{name}: tensor shape={tuple(value.shape)}, "
            f"dtype={value.dtype}, device={value.device}"
        )

        if value.numel() and torch.is_floating_point(value):
            finite = torch.isfinite(value)
            print("  All finite:", bool(finite.all().item()))

            if bool(finite.any().item()):
                v = value[finite]
                print(
                    "  Value range:",
                    float(v.min().item()),
                    "to",
                    float(v.max().item())
                )

        elif value.numel():
            unique = torch.unique(value.detach().cpu())
            print("  Unique values (up to 30):", unique[:30].tolist())

    elif isinstance(value, dict):
        print(f"{name}: dictionary keys={list(value.keys())}")
        for key, item in value.items():
            inspect_value(f"{name}[{key!r}]", item)

    elif isinstance(value, (list, tuple)):
        print(f"{name}: {type(value).__name__}, length={len(value)}")
        for i, item in enumerate(value):
            inspect_value(f"{name}[{i}]", item)

    else:
        print(f"{name}: type={type(value).__name__}, value={str(value)[:150]}")

print("\n--- BATCH STRUCTURE ---")
inspect_value("batch", batch)

# ------------------------------------------------------------
# 3. Extract the input tensor only when the structure is clear
# ------------------------------------------------------------

if isinstance(batch, dict):
    input_keys = [
        "image", "images", "input", "inputs",
        "volume", "data", "x"
    ]
    input_tensor = next(
        (
            batch[k] for k in input_keys
            if k in batch and torch.is_tensor(batch[k])
        ),
        None
    )
elif isinstance(batch, (list, tuple)) and len(batch) >= 2:
    input_tensor = batch[0] if torch.is_tensor(batch[0]) else None
else:
    input_tensor = None

if input_tensor is None:
    raise RuntimeError(
        "Input tensor could not be identified safely. "
        "Check the batch structure printed above and update the input key."
    )

# ------------------------------------------------------------
# 4. Run one forward pass without gradients
# ------------------------------------------------------------

try:
    model_device = next(model.parameters()).device
except StopIteration:
    model_device = input_tensor.device

input_tensor = input_tensor.to(model_device)

previous_training_state = model.training
model.eval()

try:
    with torch.no_grad():
        model_output = model(input_tensor)
finally:
    model.train(previous_training_state)

print("\n--- MODEL OUTPUT ---")
inspect_value("model_output", model_output)

# ------------------------------------------------------------
# 5. Completion message
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 32 INSPECTION COMPLETED")
print("No optimizer step or parameter update was performed.")
print("No metric scores were calculated from assumed label mappings.")
print("=" * 70)


In [ ]:

# STEP 33: MODEL OUTPUT ADAPTER

import numpy as np
import torch


def step33_prediction_to_regions(
    output,
    output_format,
    channel_order=("WT", "TC", "ET"),
    thresholds=(0.5, 0.5, 0.5),
    batch_index=0,
):
    """
    Accepts:
      (C,D,H,W) or (B,C,D,H,W), where C=3.
    Returns:
      {"WT": 3D mask, "TC": 3D mask, "ET": 3D mask}
    """

    if isinstance(output, (tuple, list)):
        raise TypeError(
            "Model returned multiple outputs. Select the correct "
            "segmentation output before calling this function."
        )

    if isinstance(output, dict):
        raise TypeError(
            "Model returned a dictionary. Select the correct output key first."
        )

    if torch.is_tensor(output):
        output = output.detach().float().cpu().numpy()

    output = np.asarray(output)

    if not np.isfinite(output).all():
        raise ValueError("Model output contains NaN or infinity.")

    if output.ndim == 5:
        if not 0 <= batch_index < output.shape[0]:
            raise IndexError("Invalid batch_index.")
        output = output[batch_index]

    if output.ndim != 4 or output.shape[0] != 3:
        raise ValueError(
            f"Expected (3,D,H,W) or (B,3,D,H,W); got {output.shape}"
        )

    if set(channel_order) != {"WT", "TC", "ET"}:
        raise ValueError("channel_order must contain WT, TC and ET.")

    if len(thresholds) != 3:
        raise ValueError("Provide three thresholds.")

    masks = {}

    for channel_idx, region in enumerate(channel_order):
        values = output[channel_idx]
        threshold = float(thresholds[channel_idx])

        if output_format == "logits":
            if not 0 < threshold < 1:
                raise ValueError("Thresholds must be between 0 and 1.")
            mask = values >= np.log(threshold / (1 - threshold))

        elif output_format == "probabilities":
            if values.min() < 0 or values.max() > 1:
                raise ValueError("Probabilities must be within [0,1].")
            mask = values >= threshold

        elif output_format == "binary":
            if not np.isin(values, [0, 1]).all():
                raise ValueError("Binary output must contain only 0 and 1.")
            mask = values.astype(bool)

        else:
            raise ValueError(
                "output_format must be logits, probabilities or binary."
            )

        masks[region] = mask.astype(np.uint8)

    return masks


print("Step 33 adapter defined.")


In [ ]:

# STEP 34: SHAPE, MASK AND SPACING VALIDATION

import numpy as np


def step34_verify_regions(predictions, targets, spacing):
    required = ("WT", "TC", "ET")

    if len(spacing) != 3 or any(float(s) <= 0 for s in spacing):
        raise ValueError("spacing must contain three positive voxel sizes.")

    for name, regions in (("prediction", predictions), ("target", targets)):
        if not isinstance(regions, dict):
            raise TypeError(f"{name} must be a dictionary.")

        for region in required:
            if region not in regions:
                raise KeyError(f"{name} is missing {region}.")

            mask = np.asarray(regions[region])

            if mask.ndim != 3:
                raise ValueError(f"{name}/{region} must be 3D.")

            if not np.isin(mask, [0, 1]).all():
                raise ValueError(f"{name}/{region} must be binary.")

    for region in required:
        if np.shape(predictions[region]) != np.shape(targets[region]):
            raise ValueError(
                f"Shape mismatch for {region}: "
                f"{np.shape(predictions[region])} vs "
                f"{np.shape(targets[region])}"
            )

    for name, regions in (("prediction", predictions), ("target", targets)):
        wt = np.asarray(regions["WT"], dtype=bool)
        tc = np.asarray(regions["TC"], dtype=bool)
        et = np.asarray(regions["ET"], dtype=bool)

        if np.any(tc & ~wt) or np.any(et & ~tc):
            raise ValueError(
                f"{name} masks violate ET ⊆ TC ⊆ WT."
            )

    return True


print("Step 34 verification function defined.")


In [ ]:

# STEP 35: PATIENT-LEVEL METRICS

import pandas as pd
from pathlib import Path
import numpy as np

STEP35_CSV = Path("/kaggle/working/step35_patient_metrics.csv")


def step35_evaluate_patient(
    patient_id,
    predictions,
    targets,
    spacing,
    csv_path=STEP35_CSV,
):
    step34_verify_regions(predictions, targets, spacing)

    results = calculate_wt_tc_et_metrics(
        predictions=predictions,
        targets=targets,
        spacing=spacing,
    )

    rows = []

    for region in ("WT", "TC", "ET"):
        metrics = results[region]

        rows.append({
            "patient_id": str(patient_id),
            "region": region,
            "dice": metrics.get("dice", np.nan),
            "hd95": metrics.get("hd95", np.nan),
            "iou": metrics.get("iou", np.nan),
            "precision": metrics.get("precision", np.nan),
            "recall_sensitivity": metrics.get(
                "recall_sensitivity", metrics.get("recall", np.nan)
            ),
            "specificity": metrics.get("specificity", np.nan),
        })

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(rows)
    df.to_csv(
        csv_path,
        mode="a",
        header=not csv_path.exists(),
        index=False,
    )

    print(df.to_string(index=False))
    print(f"Saved to: {csv_path}")

    return results, df


print("Step 35 patient evaluation function defined.")


In [ ]:

# STEP 36: FULL VALIDATION LOOP
# Requires Step 33 and Step 34 functions.

from pathlib import Path
import pandas as pd
import numpy as np
import torch


def step36_run_validation(
    model,
    val_loader,
    batch_adapter,
    output_format,
    output_csv="/kaggle/working/step36_validation_metrics.csv",
    channel_order=("WT", "TC", "ET"),
    thresholds=(0.5, 0.5, 0.5),
):
    """
    batch_adapter(batch) must return:
        images: tensor shaped (B,C,D,H,W)
        target_regions: list of per-patient WT/TC/ET dictionaries
        patient_ids: list of IDs
        spacings: list of (s1,s2,s3) voxel spacings

    Confirm the target label encoding and spacing before using.
    """

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    all_rows = []

    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")

    previous_mode = model.training
    model.eval()

    try:
        with torch.inference_mode():
            for batch_number, batch in enumerate(val_loader):
                images, target_regions, patient_ids, spacings = (
                    batch_adapter(batch)
                )

                images = images.to(device)
                model_output = model(images)

                if isinstance(model_output, (tuple, list, dict)):
                    raise TypeError(
                        "Select the segmentation tensor from the model "
                        "output before running Step 36."
                    )

                batch_size = images.shape[0]

                if not (
                    len(target_regions) == len(patient_ids)
                    == len(spacings) == batch_size
                ):
                    raise ValueError(
                        "Batch adapter returned inconsistent batch lengths."
                    )

                for i in range(batch_size):
                    pred_regions = step33_prediction_to_regions(
                        model_output,
                        output_format=output_format,
                        channel_order=channel_order,
                        thresholds=thresholds,
                        batch_index=i,
                    )

                    step34_verify_regions(
                        pred_regions,
                        target_regions[i],
                        spacings[i],
                    )

                    metrics = calculate_wt_tc_et_metrics(
                        predictions=pred_regions,
                        targets=target_regions[i],
                        spacing=spacings[i],
                    )

                    for region in ("WT", "TC", "ET"):
                        m = metrics[region]
                        all_rows.append({
                            "patient_id": str(patient_ids[i]),
                            "region": region,
                            "dice": m.get("dice", np.nan),
                            "hd95": m.get("hd95", np.nan),
                            "iou": m.get("iou", np.nan),
                            "precision": m.get("precision", np.nan),
                            "recall_sensitivity": m.get(
                                "recall_sensitivity",
                                m.get("recall", np.nan),
                            ),
                            "specificity": m.get("specificity", np.nan),
                        })

                print(
                    f"Validated batch {batch_number + 1}/"
                    f"{len(val_loader)}"
                )

    finally:
        model.train(previous_mode)

    if not all_rows:
        raise RuntimeError("Validation produced no patient metrics.")

    results_df = pd.DataFrame(all_rows)
    results_df.to_csv(output_csv, index=False)

    print("\nFull validation finished.")
    print("Patients:", results_df["patient_id"].nunique())
    print("CSV:", output_csv)

    return results_df


print("Step 36 validation loop defined.")
print("Do not call it until batch_adapter is written for your dataset.")


In [ ]:

# STEP 36: FULL VALIDATION LOOP
# Requires Step 33 and Step 34 functions.

from pathlib import Path
import pandas as pd
import numpy as np
import torch


def step36_run_validation(
    model,
    val_loader,
    batch_adapter,
    output_format,
    output_csv="/kaggle/working/step36_validation_metrics.csv",
    channel_order=("WT", "TC", "ET"),
    thresholds=(0.5, 0.5, 0.5),
):
    """
    batch_adapter(batch) must return:
        images: tensor shaped (B,C,D,H,W)
        target_regions: list of per-patient WT/TC/ET dictionaries
        patient_ids: list of IDs
        spacings: list of (s1,s2,s3) voxel spacings

    Confirm the target label encoding and spacing before using.
    """

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    all_rows = []

    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")

    previous_mode = model.training
    model.eval()

    try:
        with torch.inference_mode():
            for batch_number, batch in enumerate(val_loader):
                images, target_regions, patient_ids, spacings = (
                    batch_adapter(batch)
                )

                images = images.to(device)
                model_output = model(images)

                if isinstance(model_output, (tuple, list, dict)):
                    raise TypeError(
                        "Select the segmentation tensor from the model "
                        "output before running Step 36."
                    )

                batch_size = images.shape[0]

                if not (
                    len(target_regions) == len(patient_ids)
                    == len(spacings) == batch_size
                ):
                    raise ValueError(
                        "Batch adapter returned inconsistent batch lengths."
                    )

                for i in range(batch_size):
                    pred_regions = step33_prediction_to_regions(
                        model_output,
                        output_format=output_format,
                        channel_order=channel_order,
                        thresholds=thresholds,
                        batch_index=i,
                    )

                    step34_verify_regions(
                        pred_regions,
                        target_regions[i],
                        spacings[i],
                    )

                    metrics = calculate_wt_tc_et_metrics(
                        predictions=pred_regions,
                        targets=target_regions[i],
                        spacing=spacings[i],
                    )

                    for region in ("WT", "TC", "ET"):
                        m = metrics[region]
                        all_rows.append({
                            "patient_id": str(patient_ids[i]),
                            "region": region,
                            "dice": m.get("dice", np.nan),
                            "hd95": m.get("hd95", np.nan),
                            "iou": m.get("iou", np.nan),
                            "precision": m.get("precision", np.nan),
                            "recall_sensitivity": m.get(
                                "recall_sensitivity",
                                m.get("recall", np.nan),
                            ),
                            "specificity": m.get("specificity", np.nan),
                        })

                print(
                    f"Validated batch {batch_number + 1}/"
                    f"{len(val_loader)}"
                )

    finally:
        model.train(previous_mode)

    if not all_rows:
        raise RuntimeError("Validation produced no patient metrics.")

    results_df = pd.DataFrame(all_rows)
    results_df.to_csv(output_csv, index=False)

    print("\nFull validation finished.")
    print("Patients:", results_df["patient_id"].nunique())
    print("CSV:", output_csv)

    return results_df


print("Step 36 validation loop defined.")
print("Do not call it until batch_adapter is written for your dataset.")


In [ ]:

# STEP 37: OVERALL VALIDATION SUMMARY

from pathlib import Path
import pandas as pd


def step37_summarize_validation(
    metrics_csv="/kaggle/working/step36_validation_metrics.csv",
    summary_csv="/kaggle/working/step37_validation_summary.csv",
):
    df = pd.read_csv(metrics_csv)

    required = {
        "patient_id", "region", "dice", "hd95", "iou",
        "precision", "recall_sensitivity", "specificity"
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Metrics CSV is missing columns: {sorted(missing)}")

    metric_columns = [
        "dice", "hd95", "iou", "precision",
        "recall_sensitivity", "specificity"
    ]

    # Ensure invalid numeric strings become NaN.
    for column in metric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    summary = (
        df.groupby("region")[metric_columns]
        .mean()
        .reset_index()
    )

    counts = (
        df.groupby("region")["patient_id"]
        .nunique()
        .rename("patient_count")
        .reset_index()
    )

    summary = summary.merge(counts, on="region", how="left")

    Path(summary_csv).parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(summary_csv, index=False)

    print("\nRegion-wise mean metrics:")
    print(summary.to_string(index=False))
    print(f"\nSummary saved to: {summary_csv}")

    return summary


print("Step 37 summary function defined.")


In [ ]:

# STEP 38: CHECKPOINT SAVE / RESUME HELPERS

import torch
import json
import csv
from pathlib import Path

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_10parts")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def step38_save_checkpoint(
    model,
    optimizer,
    epoch,
    history,
    filename="latest_checkpoint.pt",
    scheduler=None,
    scaler=None,
    best_metric=None,
):
    checkpoint_path = CHECKPOINT_DIR / filename

    state = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": (
            optimizer.state_dict() if optimizer is not None else None
        ),
        "history": history,
        "best_metric": best_metric,
    }

    if scheduler is not None:
        state["scheduler_state_dict"] = scheduler.state_dict()

    if scaler is not None:
        state["scaler_state_dict"] = scaler.state_dict()

    temp_path = checkpoint_path.with_suffix(".tmp")
    torch.save(state, temp_path)
    temp_path.replace(checkpoint_path)

    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path


def step38_load_checkpoint(
    checkpoint_path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location=None,
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    if map_location is None:
        map_location = next(model.parameters()).device

    checkpoint = torch.load(
        checkpoint_path,
        map_location=map_location,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    next_epoch = int(checkpoint["epoch"]) + 1
    history = checkpoint.get("history", [])
    best_metric = checkpoint.get("best_metric")

    print(f"Checkpoint loaded: {checkpoint_path}")
    print(f"Resume from epoch: {next_epoch}")

    return next_epoch, history, best_metric


def step38_save_history(history, filename="training_history.json"):
    path = CHECKPOINT_DIR / filename

    with open(path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, default=str)

    print(f"Training history saved: {path}")
    return path


print("Step 38 checkpoint helpers defined.")
print("Call save/load from your existing training loop; do not start training here.")
